In [ ]:
import subprocess
import sys
from pathlib import Path

def find_wheels_dir():
    base = Path("/kaggle/input")
    for p in base.rglob("pydicom-*.whl"):
        return p.parent
    return None

wheels_dir = find_wheels_dir()
if wheels_dir is not None:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", str(wheels_dir),
            "pydicom",
        ],
        check=True,
    )
else:
    print("wheels_dir de pydicom nao encontrado -- assumindo que ja esta pre-instalado")


In [ ]:
from pathlib import Path
from IPython.display import Image, display


def _find_cover(name="RSNA_KNEE_1.png"):
    """Localiza a imagem de capa onde quer que o dataset dela tenha sido montado.

    Um caminho de montagem fixo, protegido por `exists()`, e o pior dos dois mundos:
    se errar o caminho, a imagem simplesmente nao aparece, sem nenhum aviso. O Kaggle
    tambem nem sempre monta um dataset na mesma profundidade. Pesquisar os inputs
    anexados custa uma listagem de diretorio por input e nao pode falhar em silencio.
    A montagem da competicao e pulada por inspecao em vez de por nome, porque ela guarda
    centenas de milhares de arquivos e nenhum deles e este.
    """
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for d in sorted(p for p in base.iterdir() if p.is_dir()):
        if any((d / s).is_dir() for s in ("train_series", "test_series")):
            continue
        hit = next(d.rglob(name), None)
        if hit is not None:
            return hit
    return None


_cover = _find_cover()
if _cover is not None:
    display(Image(filename=str(_cover)))


# Doze achados de uma unica RM de joelho

Cada estudo e um conjunto de series de RM adquiridas em uma sessao, e a tarefa e dar a ele
doze probabilidades: lesao do ligamento cruzado anterior e do ligamento colateral medial,
ruptura meniscal medial e lateral, osteoartrite em cada um dos tres compartimentos, derrame
articular, sinovite, cisto de Baker, contusao ossea e fratura.

As secoes estao ordenadas pela forma como as decisoes se restringem mutuamente: o que a
metrica recompensa decide como as predicoes sao combinadas, de onde vem os alvos decide o
que pode ser treinado, e o que o aparelho registrou decide o que e mostrado ao encoder.


> **Sobre as duas fontes de rotulo.** A secao 2 descreve dois "leitores" para uma mesma
> tarefa: um extrator baseado em regras, definido por completo abaixo, e um modelo de
> linguagem lendo os mesmos laudos, cuja saida e uma tabela publica anexada como dataset.
> Com a tabela montada, ela fornece os alvos; sem ela, o extrator fornece; e toda celula
> depois disso permanece inalterada.


## 1. O que a metrica recompensa

A metrica e a media nao ponderada de doze AUC-ROC, uma por rotulo:

$$\text{Score} \;=\; \frac{1}{12}\sum_{i=0}^{11} \mathrm{AUC}_i .$$

Tres consequencias decorrem disso, e cada uma elimina uma decisao de design.

**So a ordem importa.** $\mathrm{AUC}_i$ e invariante sob qualquer mapeamento estritamente
crescente dos scores do rotulo $i$, entao calibracao e limiares nao valem nada. Isso tambem
fixa como combinar modelos: fazer a media das probabilidades brutas deixa o modelo mais
confiante dominar, enquanto fazer a media dos *ranks* combina exatamente a unica informacao
que a metrica le.

**Todo rotulo custa o mesmo.** Seja $M$ a AUC media que um bom modelo conseguiria alcancar.
Um rotulo deixado no acaso contribui com $0.5$ em vez de aproximadamente $M$, perdendo
$(M-0.5)/12$ do score final por melhor que os outros onze se saiam -- com $M = 0.85$, isso e
$0.029$. Achados raros merecem *mais* atencao que os comuns, porque um achado raro e onde um
modelo mais facilmente acaba no acaso.

**Desvios de prevalencia sao sobreviveis, limiares nao.** A AUC e, em expectativa, invariante
a taxa de positivos, e a competicao declara que a prevalencia nao tem garantia de ser igual
entre os conjuntos de treino, publico e final. Um unico corte sobrevive abaixo -- os alvos
graduados sao binarizados no ponto medio para que uma AUC em holdout possa ser calculada -- e
ele decide qual epoca e configuracao sao mantidas, nunca um score submetido.


## 2. De onde vem os alvos

Apenas um pequeno subconjunto dos estudos de treino carrega os doze rotulos por condicao.
Todo estudo de treino carrega o laudo radiologico original, e a descricao dos dados convida a
derivar rotulos a partir dele.

O fato decisivo esta nos esquemas, nao na prosa: `train.csv` tem uma coluna `Report` e
`test.csv` nao tem. O texto esta disponivel no ajuste do modelo e ausente na predicao. Isso
descarta um modelo de fusao com um ramo de texto -- na inferencia ele nao teria nada para
ler -- e deixa os laudos utilizaveis apenas como alvo de treino, como sinal auxiliar
descartado na inferencia, ou como peso de quao confiavelmente um estudo pode ser lido. Este
notebook adota a primeira e a terceira: um extrator de regras multilingue le cada laudo
clausula por clausula, decidindo para cada achado se a clausula afirma, nega ou hesita sobre
ele, e emite um score com uma confianca. A confianca vira um peso de amostra, entao um estudo
cujo laudo nao diz nada sobre sinovite puxa aquela cabeca de saida muito menos do que um que a
menciona.

**Dois leitores.** Um lexico casa morfologia, entao seu modo de falha e o silencio em vez do
erro, e silencio e mensuravel sem gabarito: para cada par (laudo, achado), pergunta-se apenas
se algo deu match. Essa taxa nao precisa de anotacoes, entao roda em todo estudo, e marcada
por idioma diz *onde* o vocabulario e fraco. Aqui as falhas se concentram -- um idioma e
coberto muito melhor que os outros oito, e a lacuna cai justamente em achados que um laudo de
joelho quase sempre comenta. Enumerar morfologia para nove idiomas e o instrumento errado
para isso; ler a frase e o certo, e um modelo de linguagem le, questionado sobre os mesmos
doze achados na mesma forma graduada. Os estudos anotados decidem entre os leitores, pareados
nos mesmos estudos, e a diferenca e grande e unilateral na direcao que a taxa de cobertura
prevê. Entao o pipeline prefere uma tabela montada de rotulos lidos por modelo quando
presente e roda o lexico quando nao; ambos emitem as mesmas colunas, e uma tabela parcial cai
para o fallback por estudo, nao por execucao.

Dois detalhes importam mais que os detalhes internos de qualquer um dos leitores.

**Laudos sao graduados, anotacoes sao limiarizadas.** O radiologista que escreve o laudo e o
anotador nao compartilham um limiar. Um laudo dizendo *pequeno derrame articular* pode estar
contra uma anotacao negativa, porque o anotador marcou apenas derrames que julgou
significativos. Uma regra do tipo *termo presente $\Rightarrow$ positivo* esta errada por
construcao; graduar a mencao -- traco, sem qualificacao, marcado -- e certo e nao custa nada,
porque a secao 1 estabeleceu que so a ordem e lida.

**Rotulos derivados nao sao independentes entre estudos.** Um laudo compartilhado
literalmente por varios estudos produz um unico vetor-alvo para todos eles, o que precisa ser
respeitado ao dividir os conjuntos; a secao 7 faz isso.


### Lendo um laudo em nove idiomas

**Nenhum idioma e identificado.** Todo lexico de pistas carrega os nove idiomas de uma vez e
cada clausula e testada contra a uniao. Rotear primeiro significa se comprometer com um
palpite antes de ler qualquer evidencia, e o palpite barato -- testes de substring, `'the '`
para ingles, `'la '` para frances -- falha feio, porque `la` e tao comum em espanhol quanto
em frances e qualquer teste que rode primeiro engole os dois. Agrupar tudo custa pouco, ja
que pistas em grego e cirilico nao podem colidir com as de escrita latina e os vocabularios
de escrita latina de interesse sao proximos o bastante para que uma pista compartilhada
costume estar certa. O preco e pago em cobertura, nao em precisao.

**Normaliza, depois segmenta, depois delimita o escopo.** Caixa, diacriticos e separadores
sao dobrados primeiro, o que tambem conserta um problema de codepoint: muitos laudos em grego
escrevem mu com o MICRO SIGN U+00B5 em vez de U+03BC, e o NFKD mapeia um no outro. O texto e
entao dividido em clausulas, com uma linha de cabecalho anexada ao valor abaixo dela, porque
um laudo que le `Fractures :` e depois `Aucune.` afirma uma coisa em duas linhas, e qualquer
metodo que as separe le uma negacao como um positivo.

**Afirmacao, negacao, hesitacao.** Negacao nao e um caso de borda: para varios achados a
maioria das mencoes e negativa, ja que um laudo lista o que foi checado e encontrado intacto.
Normalidade explicita conta como negacao -- *ligamentos cruzados y colaterales dentro de
limites normales* e evidencia de ausencia, nao ausencia de evidencia -- exceto onde uma
ruptura ou um grau alto e citado na mesma frase.

**Radicais por tras das frases.** Quatro alvos precisam de uma palavra de anatomia e uma
palavra de patologia juntas. Um lexico de frases completas cobre a maioria dos matches mas
nao sobrevive a morfologia: o turco sufixa possessivos no substantivo, o croata e o grego o
declinam. Onde a frase falha, uma segunda passada casa um radical e exige um qualificador de
lado dentro de uma janela de *caracteres*, o que lida com flexao sem enumera-la, e lida com
ordem de palavras -- que coloca o adjetivo de lado antes do substantivo em ingles e depois
dele em grego.

**Por que a cobertura decide e a concordancia nao.** Uma regra que nunca dispara nao levanta
erro nenhum; em um extrator binario ela emite um negativo, indistinguivel de um negativo
confiante, e um lexico completo em ingles e fraco em grego parece um corpus onde pacientes
gregos tem menos achados. A checagem obvia -- concordancia com as anotacoes por condicao --
mede a coisa certa em estudos de menos. O erro-padrao de Hanley-McNeil de uma AUC $A$ com
$n_p$ positivos e $n_n$ negativos,

$$
\mathrm{SE}(A)=\sqrt{\frac{A(1-A)+(n_p-1)(Q_1-A^{2})+(n_n-1)(Q_2-A^{2})}{n_p\,n_n}},
\qquad
Q_1=\frac{A}{2-A},\quad Q_2=\frac{2A^{2}}{1+A},
$$

em $A\approx0.8$ e um punhado de positivos entre algumas dezenas de estudos cai perto de
$0.09$ -- um intervalo de 95% de aproximadamente $\pm0.17$ -- muito mais largo que as
diferencas entre dois lexicos que ela deveria separar. A taxa de silencio nao tem esse
limite, entao e ela que aponta para o vocabulario faltante e e nela que mudancas no lexico
sao julgadas.


In [ ]:
from __future__ import annotations

import re
import unicodedata

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]


# O i turco com/sem ponto precisa ser dobrado antes do casefold, senao "İZLENMEZ"
# e "izlenmez" divergem. O mesmo vale para ss e o d-com-tracejado croata/servio.
_PRE = str.maketrans({
    "ı": "i", "İ": "i", "I": "i", "ß": "ss", "đ": "d", "Đ": "d",
    "ø": "o", "Ø": "o", "æ": "ae", "Æ": "ae",
})


def normalize(text: str) -> str:
    """Dobra caixa, diacriticos e separadores; mantem letras gregas e cirilicas.

    A decomposicao NFKD remove acentos latinos e o tonos grego igualmente (ά -> α),
    que e o que queremos: os laudos sao inconsistentes quanto a acentos. Tambem mapeia o
    MICRO SIGN U+00B5 para um mu de verdade, o que importa porque a maioria dos laudos em
    grego aqui usa o codepoint errado.
    """
    if not isinstance(text, str):
        return ""
    text = text.translate(_PRE).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("­", "")                    # hifen suave (soft hyphen)
    text = re.sub(r"[_\-/\\]+", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text


_SENT_SPLIT = re.compile(r"(?<=[.;!?])\s+|\n+")


def clauses(text: str):
    """Divide em clausulas, depois anexa linhas `cabecalho:` ao valor que vem a seguir.

    Uma linha de laudo lendo `Fractures :` seguida de `Aucune.` e uma unica afirmacao.
    Dividir so por pontuacao separa a anatomia da sua negacao e inverte o rotulo.
    """
    norm = normalize(text)
    raw = [c.strip() for c in _SENT_SPLIT.split(norm) if c and c.strip()]

    merged = []
    for i, c in enumerate(raw):
        # Um fragmento terminado em dois-pontos e um cabecalho para o proximo fragmento.
        # Laudos estruturados em ingles escrevem cabecalhos longos - "lateral compartment
        # (meniscus, collateral ligament complex, cartilage):" tem oito palavras - entao o
        # limite e generoso.
        #
        # Um cabecalho mesclado NAO pode tambem ficar sozinho. Sozinho ele carrega a
        # palavra de anatomia sem nenhuma negacao no escopo, entao `Fractures :` /
        # `Aucune.` afirmava uma fratura a partir do cabecalho isolado enquanto a clausula
        # unida lia corretamente a negacao. A clausula unida e um superconjunto do
        # cabecalho, entao nada se perde ao descarta-lo; um cabecalho sem valor abaixo dele
        # nao e mesclado e continua valendo sozinho.
        if c.endswith(":") and len(c.split()) <= 14 and i + 1 < len(raw):
            merged.append(c + " " + raw[i + 1])
        else:
            merged.append(c)
    # Enumeracoes separadas por virgula dentro de uma clausula longa escondem afirmacoes
    # separadas.
    out = []
    for c in merged:
        out.append(c)
        if len(c.split()) > 25:
            out.extend(p.strip() for p in c.split(",") if len(p.split()) > 2)
    return out


def _rx(*alts: str) -> re.Pattern:
    return re.compile("|".join(alts))


In [ ]:
NEGATION = _rx(
    # en
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b", r"\babsence\b",
    r"\bno evidence\b", r"\bunremarkable\b", r"\bfree of\b", r"\bnone\b", r"\bnil\b",
    # es
    r"\bsin\b", r"\bno hay\b", r"\bausencia\b", r"\bausentes?\b",
    # fr
    r"\bpas de\b", r"\bsans\b", r"\baucune?\b", r"\babsence\b",
    # nl
    r"\bgeen\b", r"\bzonder\b", r"\bniet\b",
    # de
    r"\bkeine?\b", r"\bohne\b", r"\bnicht\b",
    # tr
    r"\byok\b", r"\byoktur\b", r"izlenmemekte", r"saptanmadi", r"\bdegil\b",
    r"gozlenmemekte", r"mevcut degil", r"eslik etmiyor", r"\bizlenmedi\b",
    # hr / sr / bs
    r"\bnema\b", r"\bbez\b", r"\bnisu\b", r"\bnije\b",
    # el (accents already stripped)
    r"\bδεν\b", r"\bχωρις\b", r"ουδεν",
    # bg / ru
    r"\bбез\b", r"\bне\b", r"липсва", r"\bняма\b",
)

NORMALITY = _rx(
    r"\bnormal", r"\bintact\b", r"\bpreserved\b", r"\bwithin normal limits\b",
    r"limites normales", r"\bconservad", r"\bintegr", r"\bnormales\b",
    r"\bdoga(l|ll)\b", r"korunmus", r"\bnormaldir\b", r"olagan",
    r"\buredn", r"\bocuvan", r"\bodrzan", r"\bintakt",
    r"φυσιολογικ", r"ακεραι",
    r"unauffallig", r"regelrecht", r"\bintakt\b",
    r"нормал", r"запазен", r"съхранен", r"\bбез особености\b",
    r"\bgaaf\b", r"\bnormaal\b",
)

UNCERTAIN = _rx(
    r"\bpossible\b", r"\bprobable\b", r"\bsuspicious\b", r"\bsuspected\b",
    r"cannot (be )?exclude", r"\bmay\b", r"\bquestionable\b", r"\bequivocal\b",
    r"\bposible\b", r"sin criterios categoricos", r"\bdudos",
    r"\bmuhtemel\b", r"\bolasi\b", r"\bsupheli\b", r"\bizlenim",
    r"\bmoguce\b", r"\bvjerojatno\b", r"\bsumnja\b",
    r"πιθαν", r"υποπτ",
    r"\bmoglich", r"\bverdachtig", r"\bfraglich", r"\bV\.a\.\b",
    r"\bвъзможно\b", r"\bвероятно\b", r"суспект",
    r"\bmogelijk\b", r"\bverdacht\b",
)

# Vocabulario de patologia compartilhado pelas regras pareadas.
TEAR = _rx(
    r"\btear", r"\btorn\b", r"\brupture", r"\bdisruption\b", r"discontinuit",
    r"\bavuls",
    r"\brotura\b", r"\broturas\b", r"\bruptura", r"\bdesgarro", r"\broto\b",
    r"\bdechirure", r"\bdechire",
    r"\bscheur", r"\bruptuur", r"gescheurd",
    r"riss(bildung|e|es)?\b", r"einriss", r"\bruptur", r"zerreiss", r"\blasion",
    r"\byirtik", r"\byirtig", r"\bkopma\b", r"butunluk kaybi", r"\brupturu\b",
    r"\bpuknuce", r"\bruptur", r"\bprekid\b", r"\bpukotin",
    r"ρηξη", r"ρηξις", r"ρηγμα",
    r"руптура", r"разкъсв", r"разрив", r"скъсв",
)

DEGEN = _rx(
    r"degenerat", r"\bmucoid\b", r"\bmyxoid\b", r"\bfray", r"\bfissur",
    r"dejeneratif", r"\bmukoid\b", r"degenerativn", r"εκφυλ", r"дегенерат",
    r"\bμυξοειδ", r"\bμυξωδ",
    r"\bmuco ?ide\b", r"aufgefasert",
)

INJURY = _rx(
    r"\binjur", r"\bsprain", r"\blesion", r"\blasion", r"\bedema\b", r"\boedema\b",
    r"\bodem\b", r"\bedem\b", r"\bοιδημα", r"\bодем", r"\bедем", r"\bstrain\b",
    r"\bhigh signal\b", r"\bsignal alteration\b", r"\bhiperintens", r"\bhyperintens",
    r"aumento de senal", r"alteracion de senal", r"cambio de senal",
    r"\bsignalanhebung", r"\bsignalalteration", r"verhoogd signaal", r"sinyal artis",
    r"αυξημενο σημα", r"повишен сигнал",
    r"\bthicken", r"\bzadebljanje\b", r"\bverdikking\b", r"\bdistenzij",
    r"\blaksite\b", r"\blaxity\b", r"\bpartial\b", r"\bparcijaln", r"\bparcial",
    r"\bpartiel", r"\bpartiell",
)


In [ ]:
ANAT = {
    "ACL": _rx(
        r"anterior cruciate", r"\bacl\b",
        r"cruzado anterior", r"\blca\b",
        r"croise anterieur",
        r"voorste kruisband", r"\bvkb\b",
        r"vorderes kreuzband", r"vorderen kreuzband", r"vordere kreuzband",
        r"on capraz", r"\bocb\b",
        r"prednji krizni", r"prednjeg krizn",
        r"προσθι[οα][^ ]* χιαστ", r"προσθιου χιαστου", r"χιαστο[^ ]* συνδεσμ",
        # "χιαστοι και πλαγιοι συνδεσμοι" separa o adjetivo do substantivo, entao o
        # radical do adjetivo tem que valer sozinho. O grego marca "cruzado" com ele sem ambiguidade.
        r"\bχιαστ\w*",
        r"предна кръстна", r"предната кръстна",
        # Plural, sem qualificador: laudos rotineiramente dao ambos os cruzados como normais
        # em uma clausula so ("Ligamentos cruzados y colaterales dentro de limites
        # normales"), entao a forma plural tem que casar sem um qualificador de lado ou a
        # clausula inteira se perde.
        r"cruciate ligaments", r"ligamentos cruzados", r"ligaments croises",
        r"kruisbanden", r"kreuzbander", r"capraz baglar", r"krizn[a-z]* ligament[a-z]*",
        r"χιαστοι συνδεσμ", r"χιαστων συνδεσμ", r"кръстните връзки", r"кръстни връзки",
    ),
    "MCL": _rx(
        r"medial collateral", r"\bmcl\b", r"tibial collateral",
        r"colateral medial", r"colateral interno", r"\blcm\b",
        r"collateral medial", r"collateral interne",
        r"mediale collaterale", r"binnenband", r"\b(mediale|laterale) banden\b",
        r"\bcollaterale banden\b",
        r"innenband", r"mediales? kollateral",
        r"\bic yan bag", r"medial kollateral", r"\biyb\b",
        r"medijalni kolateraln", r"medijalnog kolateraln",
        r"εσω πλαγι", r"εσωτερικο πλαγι", r"\bπλαγι\w* συνδεσμ", r"\bπλαγιοι\b",
        r"медиален колатерал", r"вътрешна странична", r"\bколатерал\w*",
        # Mesmo padrao de plural dos cruzados.
        # "Ligamentos cruzados y colaterales" separa o substantivo do adjetivo, entao
        # o adjetivo tem que valer sozinho como pista.
        r"\bcolaterales\b", r"\bcollateraux\b", r"\bcollateralen\b", r"\bkolateralni\b",
        r"collateral ligaments", r"ligamentos colaterales", r"ligaments collateraux",
        r"collaterale banden", r"kollateralbander", r"seitenbander", r"yan baglar",
        r"kolateraln[a-z]* ligament[a-z]*", r"πλαγιοι συνδεσμ", r"πλαγιων συνδεσμ",
        r"колатерални връзки", r"страничните връзки",
    ),
    "Medial Meniscus": _rx(
        r"medial meniscus", r"\bmm\b(?= tear)", r"medial menisc",
        r"menisco medial", r"menisco interno",
        r"menisque medial", r"menisque interne",
        r"mediale meniscus", r"binnenmeniscus",
        r"innenmeniskus", r"medialen? meniskus", r"innenmeniskushinterhorn",
        r"medyal menisk", r"\bic menisk",
        r"medijalni meniskus", r"medijalnog meniskusa", r"medijalnom meniskusu",
        r"εσω μηνισκ", r"μηνισκ[^ ]* του εσω", r"εσω διαμερισμα[^.]{0,40}μηνισκ",
        r"медиалния менискус", r"медиален менискус", r"вътрешния менискус",
    ),
    "Lateral Meniscus": _rx(
        r"lateral meniscus", r"lateral menisc",
        r"menisco lateral", r"menisco externo",
        r"menisque lateral", r"menisque externe",
        r"laterale meniscus", r"buitenmeniscus",
        r"aussenmeniskus", r"lateralen? meniskus",
        r"lateral menisk", r"\bdis menisk",
        r"lateralni meniskus", r"lateralnog meniskusa", r"lateralnom meniskusu",
        r"εξω μηνισκ", r"μηνισκ[^ ]* του εξω", r"εξω διαμερισμα[^.]{0,40}μηνισκ",
        r"латералния менискус", r"латерален менискус", r"външния менискус",
    ),
}

# Osteoartrite raramente e escrita como "osteoarthritis". E escrita como perda de
# cartilagem, grau de condropatia, estreitamento do espaco articular, ou osteofitos -
# delimitada a um compartimento.
OA_EVIDENCE = _rx(
    r"osteoarthrit", r"\barthros", r"\bgonarthros", r"\bosteoarthros",
    r"chondropath", r"chondromalac", r"condropat", r"condromalac",
    r"cartilage loss", r"cartilage thinning", r"chondral (loss|defect|ulcer|thinning)",
    r"osteophyt", r"osteofit", r"osteofyt", r"osteofito", r"osteophyten",
    r"joint space narrowing", r"pinzamiento articular",
    r"kikirdak kayb", r"kikirdak incelme", r"kondropati", r"kondral",
    r"kraakbeen(lijden|verlies)", r"gonartrose", r"artrose",
    r"knorpel(verlust|schaden|defekt)", r"arthrose", r"gonarthrose",
    r"hrskavic", r"hondromalac", r"artroz", r"osteoartrit",
    r"χονδρ[^ ]*παθ", r"αρθριτ", r"αρθρωσ", r"οστεοφυτ",
    r"αρθρικου χονδρου", r"εξαλειψη του αρθρικου χονδρου",
    r"артроз", r"хондропат", r"остеофит", r"хрущял[^.]{0,30}(изтън|увред|дефект)",
    r"ulcera[s]? condral", r"cartilago[^.]{0,25}(perdida|adelgaz)",
    r"icrs grade", r"outerbridge",
)

COMPARTMENT = {
    "Medial OA": _rx(
        r"medial (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial medial", r"femorotibial interno",
        r"mediaal femorotibiaal", r"mediale femorotibial",
        r"medial femorotibial", r"medialen kompartiment", r"innere[sn]? kompartiment",
        r"medyal femorotibial", r"ic kompartman", r"medyal kompartman",
        r"medijaln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εσω διαμερισμα", r"εσω κνημιαι", r"εσω μηριαι",
        r"медиалн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"medial (femoral|tibial) (condyle|plateau)", r"condilo femoral medial",
        r"medialen? (femurkondyl|tibiaplateau)", r"mediale femorale condyl",
    ),
    "Lateral OA": _rx(
        r"lateral (femorotibial|tibiofemoral|compartment)",
        r"compartimento femorotibial lateral", r"femorotibial externo",
        r"lateraal femorotibiaal", r"laterale femorotibial",
        r"lateral femorotibial", r"lateralen kompartiment", r"aussere[sn]? kompartiment",
        r"lateral femorotibial", r"dis kompartman", r"lateral kompartman",
        r"lateraln[^ ]* (femorotibi|odjelj|kompartm)",
        r"εξω διαμερισμα", r"εξω κνημιαι", r"εξω μηριαι",
        r"латералн[^ ]* (компартм|отдел|тибиал|феморотиб)",
        r"lateral (femoral|tibial) (condyle|plateau)", r"condilo femoral lateral",
        r"lateralen? (femurkondyl|tibiaplateau)", r"laterale femorale condyl",
    ),
    "PF OA": _rx(
        r"patellofemoral", r"femoropatellar", r"femoropatelar", r"patelofemoral",
        r"retropatellar", r"retrorotulian", r"\btrochlea", r"\btroclea", r"\btroklea",
        r"\bpatella\b", r"\bpatellar\b", r"\brotulian", r"\brotula\b", r"\bpatele\b",
        r"\bpatellae?\b", r"patellofemoraal", r"femoropatellair",
        r"επιγονατιδ", r"μηροεπιγονατιδ", r"τροχιλ",
        r"пател", r"феморопател", r"тролх",
        r"anterior compartment", r"compartimento anterior", r"prednj[^ ]* odjeljk",
    ),
}

# Achados autodeclarados: o termo em si e o achado.
DIRECT = {
    "Effusion": _rx(
        r"\beffusion", r"joint fluid", r"intra ?articular fluid", r"\bhydrops\b",
        r"derrame articular", r"\bderrame\b", r"liquido articular",
        r"epanchement",
        r"gewrichtsvocht", r"\bvocht\b", r"\bhydrops\b", r"gewrichtseffusie",
        r"gelenkerguss", r"\berguss\b", r"gelenksergu",
        # "diz eklemi ici sivi miktari ... artmis" e "eklem icerisinde yaygin sivi
        # artisi" ocorrem os dois; o substantivo recebe um sufixo possessivo, entao
        # `eklem ` sozinho perde o match. Casa o radical mais qualquer sufixo.
        r"eklem\w* ic\w* sivi", r"efuzyon", r"eklem sivisi",
        r"sivi (miktari|artisi|birikimi)", r"sivi artis", r"\bsivi\b[^.]{0,25}artmis",
        r"\bizljev", r"\bizliv", r"zglobn[^ ]* tekucin", r"\bhidrops\b",
        r"αρθρικ[^ ]* υγρ", r"υγρου ενδαρθρικα", r"ενδαρθρικ[^ ]* υγρ", r"ποσοτητα υγρου",
        r"ενδαρθρικ", r"αρθρικη συλλογη", r"υγρο στην αρθρωση", r"υγρου στην αρθρωση",
        r"ставен излив", r"излив", r"ставна течност", r"синовиална течност",
    ),
    "Synovitis": _rx(
        r"synovit", r"sinovit", r"synovial (thickening|proliferation|hypertroph)",
        r"synovitis", r"synoviale? (verdikking|proliferatie)",
        r"synovialitis", r"synovialis(verdickung|proliferation)",
        r"sinovijalitis", r"sinovitis", r"zadebljanje sinovij",
        r"υμενιτιδα", r"συνοβιτιδα", r"υμενικ[^ ]* υπερτροφ", r"αρθρικου υμεν",
        r"синовит", r"синовиал[^ ]* (задебел|пролифер)",
        r"verdikkingen van (het )?synovium", r"pannus",
    ),
    "Baker's": _rx(
        r"baker", r"popliteal cyst", r"quiste popliteo", r"quistes popliteos",
        r"kyste poplite", r"popliteale? cyst", r"poplitealzyste", r"bakerzyste",
        r"popliteal kist", r"\bbakerova\b", r"poplitealn[^ ]* cist",
        r"κυστη baker", r"πολυχωρη συνοβιακη κυστη", r"κυστη του baker",
        r"киста на бейкър", r"бейкърова киста", r"поплитеална киста",
        r"gastrocnemio ?semimembranos", r"gastrocnemius semimembranosus burs",
    ),
    "Contusion": _rx(
        r"\bcontusion", r"bone bruise", r"bone marrow (o?edema|contusion)",
        r"\bkontuz", r"medular bone o?edema", r"marrow o?edema",
        r"contusion osea", r"edema oseo", r"edema de medula osea",
        r"oedeme osseux", r"contusion osseuse",
        r"botcontusie", r"botoedeem", r"beenmergoedeem", r"botmergoedeem",
        r"knochenmarkodem", r"knochenodem", r"kontusion", r"bone bruise",
        r"kemik kontuzyonu", r"kemik iligi odemi", r"kemik odemi",
        r"kostani edem", r"edem kosti", r"kontuzij",
        r"οστεομυελικ[^ ]* οιδημα", r"οστικο οιδημα", r"μυελικο οιδημα",
        r"костномозъчен едем", r"костен едем", r"контузионен",
    ),
    "Fracture": _rx(
        r"\bfractur", r"\bfract\b",
        r"\bfractura", r"\bfracturas\b",
        r"\bfractuur", r"\bbreuk\b",
        r"\bfraktur", r"\bbruch\b",
        r"\bkirik\b", r"\bkirigi\b", r"\bkirik\b",
        r"\bfraktur", r"\bprijelom", r"impresijsk[^ ]* fraktur",
        r"καταγμα", r"καταγματ",
        r"фрактур", r"счупван", r"фисур",
        r"insufficiency fracture", r"stress fracture", r"avulsion fracture",
        r"subchondral fracture", r"subkondral kiri",
    ),
}

# Termos que parecem um achado mas nao sao o achado sendo pontuado.
DECOY = {
    # `no fracture` esta deliberadamente ausente: um decoy pula a clausula, entao listar
    # isso aqui transformava a negacao mais comum em ingles em silencio, e o estudo entao
    # puxava a cabeca de fratura com o peso de um laudo que nunca mencionou fratura
    # nenhuma. `microfractur` e um procedimento cirurgico e `fracture risk` uma predicao;
    # os dois ficam.
    "Fracture": _rx(r"microfractur", r"\bfracture (risk|prophyla)"),
    "Baker's": _rx(r"meniscal cyst", r"quiste meniscal", r"ganglion"),
}

PAIRED = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_TARGETS = {"Medial OA", "Lateral OA", "PF OA"}


In [ ]:
STEM_MENISCUS = _rx(r"menisc\w*", r"menisk\w*", r"μηνισκ\w*", r"мениск\w*")
STEM_CRUCIATE = _rx(r"cruciate", r"cruzado", r"croise", r"kruisband", r"kreuzband",
                    r"capraz bag\w*", r"krizn\w*", r"χιαστ\w*", r"кръстн\w*",
                    r"\bacl\b", r"\bpcl\b", r"\blca\b", r"\blcp\b", r"\bvkb\b",
                    r"\bhkb\b", r"\bocb\b", r"\bacb\b")
STEM_COLLATERAL = _rx(r"collateral\w*", r"colateral\w*", r"kollateral\w*",
                      r"collaterale\w*", r"kolateraln\w*", r"yan bag\w*",
                      r"πλαγι\w*", r"колатерал\w*", r"странич\w*",
                      r"innenband\w*", r"aussenband\w*", r"binnenband\w*",
                      r"\bmcl\b", r"\blcl\b", r"\blcm\b", r"\biyb\b")

SIDE_MEDIAL = _rx(r"\bmedial\w*", r"\bmedyal\w*", r"\bmedijaln\w*", r"\bmediaal\w*",
                  r"\bmediale\w*", r"\bintern[oa]\w*", r"\binterne\w*", r"\binnen\w*",
                  r"\bic\b", r"\bunutarnj\w*", r"\bεσω\w*", r"\bεσωτερικ\w*",
                  r"\bмедиал\w*", r"\bвътреш\w*", r"\btibial collateral\b",
                  r"\bbinnen\w*", r"\bmediaal\b")
SIDE_LATERAL = _rx(r"\blateral\w*", r"\bextern[oa]\w*", r"\bexterne\w*", r"\bdis\b",
                   r"\blateraln\w*", r"\baussen\w*", r"\bbuiten\w*", r"\bεξω\w*",
                   r"\bεξωτερικ\w*", r"\bлатерал\w*", r"\bвъншн\w*",
                   r"\bfibular collateral\b", r"\bvanjsk\w*")
SIDE_ANTERIOR = _rx(r"\banterior\w*", r"\bant\b", r"\bon\b", r"\bprednj\w*",
                    r"\bvorder\w*", r"\bvoorste\b", r"\bπροσθι\w*", r"\bпредн\w*",
                    r"\banteriyor\w*", r"\bavant\b", r"\bant[eé]rieur\w*")

# O oposto de SIDE_ANTERIOR, necessario apenas para impedir que uma pista de cruzado
# cega quanto ao lado dispare no ligamento posterior. Nunca e usado para afirmar um
# alvo - nao existe alvo PCL - entao e deliberadamente estreito: `posterior horn`
# (corno posterior) e uma das frases mais comuns num laudo de joelho e nao pode ser
# lida como um qualificador de cruzado, por isso a guarda abaixo testa proximidade ao
# radical do cruzado em vez de presenca na clausula.
SIDE_POSTERIOR = _rx(r"\bposterior\w*", r"\bpost[eé]rieur\w*", r"\bposteriore\w*",
                     r"\bhinter\w*", r"\bachterste\b", r"\barka\b", r"\bstraznj\w*",
                     r"\bzadnj\w*", r"\bοπισθι\w*", r"\bзадн\w*", r"\bpostero\w*")

# Fracture e o alvo cujo radical mais varia ao longo do corpus.
STEM_FRACTURE = _rx(r"fractur\w*", r"fraktur\w*", r"fractuur\w*", r"\bfract\b",
                    r"kiri[kgğ]\w*", r"prijelom\w*", r"lom kosti", r"\bbreuk\w*",
                    r"\bbruch\w*", r"καταγμα\w*", r"καταγματ\w*", r"фрактур\w*",
                    # NAO um `fissur\w*` isolado: "fisuras condrales" e "full thickness
                    # fissures in the articular cartilage" descrevem cartilagem, nao osso.
                    # O radical precisa estar ancorado a uma palavra de osso para
                    # significar fratura.
                    r"счупван\w*", r"fisur\w* (osea|oseas|kost)", r"fissur\w* kost")

STEM_OA_COMPARTMENT = _rx(r"compartment\w*", r"compartimento\w*", r"compartiment\w*",
                          r"kompartman\w*", r"kompartiment\w*", r"odjelj\w*",
                          r"διαμερισμα\w*", r"компартм\w*", r"\bотдел\w*",
                          r"femorotibial\w*", r"femorotibiaal\w*", r"tibiofemoral\w*",
                          r"femoro tibial\w*", r"κνημιαι\w*", r"μηριαι\w*",
                          r"femoral condyl\w*", r"tibial plateau\w*",
                          r"condilo femoral", r"platillo tibial", r"tibiaplateau\w*",
                          r"femurkondyl\w*", r"femoralne? kondil\w*",
                          r"tibijaln\w* plato", r"femoral kondil\w*",
                          r"tibia plato", r"tibyal plato")


def _distance(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    """Caracteres do radical mais proximo ate o qualificador mais proximo, ou None se nenhum
    estiver perto.

    Janelas de caracteres em vez de janelas de tokens, porque a ordem das palavras
    difere: o ingles coloca o lado antes do substantivo, o grego e o bulgaro costumam
    colocar depois, e o turco o anexa como um adjetivo separado que vem antes.

    Uma distancia em vez de um sim/nao. Presenca basta para decidir que um qualificador
    se aplica a uma estrutura, mas nao basta para decidir qual dos dois qualificadores se
    aplica: um laudo de joelho diz "corno anterior" e "corno posterior" o tempo todo,
    entao qualquer janela larga o bastante para pegar uma palavra de lado real tambem
    pega uma nao relacionada, e duas regras que ambas respondem "sim" nao podem ser
    distinguidas. Comparar a que distancia cada uma esta pode.
    """
    best = None
    for m in stem_rx.finditer(clause):
        lo = max(0, m.start() - window)
        hi = min(len(clause), m.end() + window)
        for q in qual_rx.finditer(clause[lo:hi]):
            qs, qe = lo + q.start(), lo + q.end()
            d = 0 if qs < m.end() and qe > m.start() else \
                min(abs(m.start() - qe), abs(qs - m.end()))
            best = d if best is None else min(best, d)
    return best


def _near(clause: str, stem_rx: re.Pattern, qual_rx: re.Pattern, window: int = 55):
    """True se um match de radical tem um qualificador dentro de `window` caracteres de
    qualquer lado."""
    return _distance(clause, stem_rx, qual_rx, window) is not None
    return False


# conceito -> pares (radical, lado) usados em complemento aos lexicos de frase acima
STEM_RULES = {
    "ACL": (STEM_CRUCIATE, SIDE_ANTERIOR),
    "MCL": (STEM_COLLATERAL, SIDE_MEDIAL),
    "Medial Meniscus": (STEM_MENISCUS, SIDE_MEDIAL),
    "Lateral Meniscus": (STEM_MENISCUS, SIDE_LATERAL),
    "Medial OA": (STEM_OA_COMPARTMENT, SIDE_MEDIAL),
    "Lateral OA": (STEM_OA_COMPARTMENT, SIDE_LATERAL),
}


In [ ]:
SEV_LOW = _rx(
    r"\bsmall\b", r"\bminimal\b", r"\btrace\b", r"\bmild\b", r"\bslight\b",
    r"\btiny\b", r"\bscant\b", r"\bmimimal\b", r"\bdiscrete\b", r"\bfocal\b",
    r"\bleve\b", r"\bminim", r"\bpeque", r"\bligero\b", r"\bescaso\b", r"\bdiscreto\b",
    r"\bhafif\b", r"\bminimal\b", r"\baz miktarda\b", r"\bsilik\b",
    r"\bmanja\b", r"\bmanji\b", r"\bblago\b", r"\bdiskretn", r"\bmalo\b",
    r"\bgering", r"\bdiskret", r"\bkleine?r?\b", r"\bwenig\b", r"\bzarte?\b",
    r"\bbeperkte?\b", r"\bgeringe\b", r"\bweinig\b", r"\blichte?\b",
    r"\bηπι", r"\bμικρ", r"\bελαχιστ",
    r"\bминимал", r"\bлек", r"\bмалк", r"\bнеголям",
)

SEV_HIGH = _rx(
    r"\blarge\b", r"\bmarked\b", r"\bmassive\b", r"\bsevere\b", r"\bextensive\b",
    r"\bmoderate\b", r"\bgross\b", r"\bsignificant\b", r"\babundant\b", r"\btense\b",
    r"\bmoderad", r"\bimportante\b", r"\bsevera?\b", r"\bmarcad", r"\bcuantios",
    r"\bbelirgin\b", r"\byaygin\b", r"\bileri\b", r"\bciddi\b", r"\bbol\b",
    r"\bopsezan\b", r"\bveliki\b", r"\bizrazit", r"\bznacajn", r"\bumjeren",
    r"\bausgepragt", r"\bdeutlich", r"\bmassiv", r"\bmassig", r"\bgross",
    r"\buitgebreid", r"\bgevorderd", r"\bveel\b", r"\bmatige?\b",
    r"\bμετρι", r"\bμεγαλ", r"\bεκτεταμεν", r"\bευμεγεθ", r"\bσοβαρ",
    r"\bголям", r"\bизразен", r"\bзначим", r"\bумерен", r"\bобилен",
)

# OA (osteoartrite) e frequentemente afirmada para a articulacao inteira em vez de por
# compartimento ("tricompartmental osteoarthritis", "gonarthrose", "incipient OA of all
# three compartments"). Essas afirmacoes sao evidencia para os tres alvos de OA.
GLOBAL_OA = _rx(
    r"tri ?compartment", r"all three compartment", r"global(ised)? (oa|osteoarthrit)",
    r"\bgonarthros", r"\bgonartros", r"\bgonarthrose", r"\bgonartrose",
    r"osteoarthritis of the knee", r"artrosis (de |)(la )?rodilla", r"knee osteoarthrit",
    r"\bdiz osteoartrit", r"\bgonartroz", r"artroza koljena",
    r"οστεοαρθριτιδα", r"αρθριτιδα του γονατος",
    r"артроза на колянната", r"гонартроз",
    r"degenerative joint disease", r"\bdjd\b",
)

# Um "edema de medula ossea" isolado nao e uma contusao quando esta sob um defeito de
# cartilagem: edema subcondral abaixo de um compartimento desgastado e sinal
# degenerativo reativo, e le-lo como uma contusao transforma todo joelho
# osteoartritico num caso de trauma.
DEGENERATIVE_MARROW = _rx(
    r"subchondral", r"subcondral", r"subkondral", r"supkondraln", r"subchondraln",
    r"υποχονδρι", r"субхондрал", r"subchondrale?",
    r"\bcyst", r"\bquist", r"\bzyste\b", r"\bcistic", r"reactive", r"reactivo",
)

TRAUMA = _rx(
    r"\bbruise\b", r"\bcontusion", r"\bkontuz", r"\bcontusion osea\b",
    r"\btrauma", r"\bimpaction\b", r"\bpivot shift\b", r"\bkissing\b",
    r"\bacute\b", r"\bagudo\b", r"\bakut", r"\bpivot kaymasi\b",
    r"\bcontusion osseuse\b", r"\bbone bruise\b", r"\bbotcontusie\b",
    r"\bконтузион", r"\bμωλωπ", r"\bkontuzij",
)


In [ ]:
def _polarity(clause: str, anchor_end: int) -> str:
    """Classifica uma clausula como positiva, negativa ou incerta para um termo casado.

    O escopo e a clausula inteira. A segmentacao de clausulas ja mantem as afirmacoes
    curtas, e uma janela em caracteres delimita mal o escopo entre idiomas com ordens de
    palavras diferentes - o turco coloca seu negador no fim da frase, o ingles no
    inicio.
    """
    if UNCERTAIN.search(clause):
        return "uncertain"
    if NEGATION.search(clause):
        return "negative"
    if NORMALITY.search(clause):
        # "meniscus normal" nega; "normal ... but tear" nao.
        if TEAR.search(clause) or re.search(r"\bgrade [34]\b", clause):
            return "positive"
        return "negative"
    return "positive"


class _Matcher:
    """Lexico de frases primeiro, proximidade radical+lado como fallback.

    Expoe `.search` para se encaixar no mesmo lugar que um padrao compilado.
    """

    def __init__(self, phrase_rx, stem=None, side=None, window=55, contrary=None):
        self.phrase_rx = phrase_rx
        self.stem = stem
        self.side = side
        self.window = window
        self.contrary = contrary

    def search(self, clause):
        m = self.phrase_rx.search(clause)
        if m is not None and not self._wrong_side(clause):
            return m
        if self.stem is not None and _near(clause, self.stem, self.side, self.window):
            return self.stem.search(clause)
        return None

    def _wrong_side(self, clause):
        """True quando a clausula nomeia o outro membro do par desta estrutura.

        Algumas pistas no lexico sao deliberadamente cegas quanto ao lado: o grego
        separa o adjetivo do substantivo ("cruciate and collateral ligaments"), entao o
        radical do adjetivo isolado tem que valer sozinho ou a clausula se perde. Esse
        radical entao tambem casa com o cruzado posterior e o colateral lateral, nenhum
        dos dois um alvo aqui, e um positivo supera qualquer negativo no scorer - entao
        uma unica clausula de PCL era suficiente para sobrepor um explicito "the ACL is
        normal".

        O teste e proximidade ao radical da propria estrutura, nao presenca na clausula.
        "Posterior horn of the medial meniscus" aparece numa grande parcela dos laudos
        de joelho e nao diz nada sobre um cruzado; so um qualificador ao lado da palavra
        do ligamento e um. Uma clausula que nomeia os dois lados mantem o match, porque
        ela de fato menciona este alvo.
        """
        if self.contrary is None or self.stem is None:
            return False
        other = _distance(clause, self.stem, self.contrary, self.window)
        if other is None:
            return False
        own = _distance(clause, self.stem, self.side, self.window)
        # Nao "o outro lado e mencionado" mas "ele e o mais proximo dos dois". Uma
        # clausula lendo "tear of the posterior horn of the medial meniscus; the
        # cruciate ligaments are intact" menciona posterior, e sob um teste de mera
        # presenca isso bastava para suprimir a pista do cruzado - o oposto da intencao,
        # ja que a clausula de fato descreve os ligamentos. Em caso de empate mantem o
        # match: uma clausula que nomeia os dois lados de fato menciona este.
        return own is None or other < own


# Qual pista, se estiver ao lado do radical da estrutura, significa que a clausula e
# sobre o outro membro do par. So as duas estruturas com pista cega quanto ao lado
# precisam de uma.
CONTRARY = {"ACL": SIDE_POSTERIOR, "MCL": SIDE_LATERAL}

ANAT_MATCH = {
    tgt: _Matcher(ANAT[tgt], *STEM_RULES[tgt], contrary=CONTRARY.get(tgt))
    for tgt in PAIRED
}
COMPARTMENT_MATCH = {
    "Medial OA": _Matcher(COMPARTMENT["Medial OA"], *STEM_RULES["Medial OA"]),
    "Lateral OA": _Matcher(COMPARTMENT["Lateral OA"], *STEM_RULES["Lateral OA"]),
    "PF OA": _Matcher(COMPARTMENT["PF OA"]),
}
DIRECT_MATCH = {
    tgt: _Matcher(_rx(rx.pattern, STEM_FRACTURE.pattern) if tgt == "Fracture" else rx)
    for tgt, rx in DIRECT.items()
}


def _severity(clause: str) -> float:
    """Pondera uma mencao positiva por quao enfatica e a frase.

    Ordenado, nao calibrado. Um "moderate effusion" precisa superar um "trace effusion"
    e ambos precisam superar o silencio; os numeros absolutos nao importam para a AUC.
    """
    high = SEV_HIGH.search(clause) is not None
    low = SEV_LOW.search(clause) is not None
    if high and not low:
        return 1.0
    if low and not high:
        return 0.45
    return 0.75                       # unqualified mention


def _score_clauses(cls, anat_rx, path_rx=None, decoy_rx=None, context_penalty=None,
                   context_bonus=None):
    """Acumula evidencia graduada sobre as clausulas para um alvo.

    Retorna (score, confidence, n_pos, n_neg). Positivos sao graduados por severidade e
    por regexes de contexto opcionais; negativos so importam quando nada positivo foi
    encontrado, porque os laudos afirmam normalidade para toda estrutura que checam.
    """
    n_pos = n_neg = n_unc = 0
    best = 0.0
    for c in cls:
        m = anat_rx.search(c)
        if not m:
            continue
        if decoy_rx is not None and decoy_rx.search(c):
            continue
        if path_rx is not None and not path_rx.search(c):
            if NORMALITY.search(c) and not NEGATION.search(c):
                n_neg += 1
            continue
        pol = _polarity(c, m.end())
        if pol == "positive":
            n_pos += 1
            w = _severity(c)
            if context_penalty is not None and context_penalty.search(c):
                w *= 0.45
            if context_bonus is not None and context_bonus.search(c):
                w = min(1.0, w * 1.35)
            best = max(best, w)
        elif pol == "negative":
            n_neg += 1
        else:
            n_unc += 1
            best = max(best, 0.30)

    if n_pos or n_unc:
        # 0.52 .. 0.95, ordenado pela mencao unica mais forte, ajustado pela repeticao.
        score = min(0.95, 0.50 + 0.42 * best + 0.03 * min(n_pos, 3))
        conf = min(1.0, 0.55 + 0.15 * n_pos)
    elif n_neg:
        score = max(0.04, 0.20 - 0.04 * n_neg)
        conf = min(0.9, 0.45 + 0.12 * n_neg)
    else:
        score, conf = 0.28, 0.05          # silencio fica acima de negativo-afirmado
    return score, conf, n_pos, n_neg


def extract(report: str) -> dict:
    """Extrai doze pares (score, confidence) de um laudo."""
    cls = clauses(report)
    out = {}
    path_paired = _rx(TEAR.pattern, DEGEN.pattern, INJURY.pattern)

    for tgt in TARGETS:
        if tgt in PAIRED:
            s, c, npos, nneg = _score_clauses(cls, ANAT_MATCH[tgt], path_paired)
        elif tgt in OA_TARGETS:
            s, c, npos, nneg = _score_clauses(cls, COMPARTMENT_MATCH[tgt], OA_EVIDENCE)
        elif tgt == "Contusion":
            # Edema subcondral reativo sob um defeito de cartilagem e osteoartrite,
            # nao uma contusao. Linguagem explicita de trauma empurra na outra direcao.
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt),
                                              context_penalty=DEGENERATIVE_MARROW,
                                              context_bonus=TRAUMA)
        else:
            s, c, npos, nneg = _score_clauses(cls, DIRECT_MATCH[tgt], None, DECOY.get(tgt))
        out[tgt] = s
        out[tgt + "__conf"] = c
        out[tgt + "__npos"] = npos
        out[tgt + "__nneg"] = nneg

    # --- correcoes entre alvos ------------------------------------------ #
    # Uma afirmacao de osteoartrite da articulacao inteira e evidencia para todo
    # compartimento que nao foi avaliado separadamente. Sem isso, "incipient OA of all
    # three compartments" pontuaria zero nos tres alvos de OA.
    g_hits = [c for c in cls if GLOBAL_OA.search(c) and _polarity(c, 0) == "positive"]
    if g_hits:
        gscore = 0.50 + 0.42 * max(_severity(c) for c in g_hits)
        for tgt in OA_TARGETS:
            if out[tgt + "__npos"] == 0 and out[tgt + "__nneg"] == 0:
                out[tgt] = max(out[tgt], gscore * 0.92)
                out[tgt + "__conf"] = max(out[tgt + "__conf"], 0.4)

    # Sinovite e frequentemente visivel nas imagens e ausente do texto, entao aqui o
    # silencio e evidencia fraca de ausencia de um jeito que nao e para outros achados.
    # Effusion e o proxy textual mais confiavel para ela - as duas compartilham um
    # mecanismo - entao uma sinovite silenciosa herda uma fracao da evidencia de effusion
    # em vez de cair pro piso.
    if out["Synovitis__npos"] == 0 and out["Synovitis__nneg"] == 0:
        out["Synovitis"] = max(out["Synovitis"], 0.28 + 0.45 * (out["Effusion"] - 0.28))

    return out


## 3. Lendo a aquisicao

`train_series.csv` descreve cada serie com um plano anatomico e duas flags binarias,
`Fluid_Sensitive` e `Fat_Suppression`. Os nomes denotam duas propriedades fisicamente
independentes -- e essa independencia e o que as colunas entregues nao tem: em toda a serie
de treino as duas concordam em toda linha, entao, como estao dadas, carregam um unico eixo em
vez de dois. Esse e o primeiro motivo para recuperar as duas a partir do cabecalho.

*Sensibilidade a fluido* e uma propriedade da **ponderacao de contraste**, definida pelo
tempo de repeticao $T_R$ e pelo tempo de eco $T_E$:

$$
\text{ponderacao} \;=\;
\begin{cases}
T_1 & T_R \lesssim 800\ \text{ms}\\
T_2 & T_R \gtrsim 800\ \text{ms},\ T_E \gtrsim 60\ \text{ms}\\
\text{DP} & T_R \gtrsim 800\ \text{ms},\ T_E \lesssim 60\ \text{ms}
\end{cases}
$$

Fluido e brilhante em $T_2$, intermediario em densidade de protons, escuro em $T_1$. Gradiente
eco quebra a regra -- seu $T_R$ e curto por design -- entao isso e resolvido primeiro por
`ScanningSequence`. *Supressao de gordura* e uma **preparacao** aplicada em cima de qualquer
ponderacao, e e o que torna o edema de medula ossea conspicuo. As duas sao lidas de
`SeriesDescription`, `SequenceName` e `ScanOptions` onde o protocolo as nomeia, e de $T_R$ e
$T_E$ onde nao nomeia.

### Quais sequencias mostrar ao modelo

Um joelho e lido em tres planos porque as estruturas correm em direcoes diferentes:
ligamentos cruzados obliquamente, melhor vistos sagitalmente; ligamentos colaterais e o corpo
do menisco coronalmente; cartilagem patelar e os retinaculos axialmente. Cruzar o plano com
os dois eixos de aquisicao da os slots abaixo, escolhidos para que cada um dos doze achados
tenha ao menos uma sequencia que o mostre bem.

| slot | plano | ponderacao | fat sat | o que carrega |
|---|---|---|---|---|
| `SAG_FLUID_FS` | sagital | DP / T2 | sim | rupturas meniscais, edema de medula, derrame |
| `COR_FLUID_FS` | coronal | DP / T2 | sim | ligamentos colaterais, corpo do menisco, edema |
| `AX_FLUID_FS` | axial | DP / T2 | sim | articulacao patelofemoral, sinovio, derrame |
| `SAG_FLUID_NOFS` | sagital | DP / T2 | nao | morfologia meniscal em alto contraste-ruido |
| `COR_T1` | coronal | T1 | nao | arquitetura da medula, contorno de cartilagem e osso |
| `SAG_T1` | sagital | T1 | nao | anatomia, alteracao cronica |

Um estudo raramente tem os seis; uma mascara de presenca por slot leva as ausencias ate a
cabeca de saida, que a secao 6 usa.


In [ ]:
from __future__ import annotations

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import json
import re
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

# O extrator de rotulos e definido nas celulas acima quando isto roda como notebook.
# Como script simples ele e importado do codigo-fonte do pacote, entao os dois
# caminhos compartilham uma unica definicao em vez de manter uma copia cada um.

T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]


# O recorte central tem que ser menor que o menor campo de visao do corpus, ou ele nao
# faz nada em silencio. Medido em toda serie de treino, o campo de visao adquirido
# (Rows x PixelSpacing) tem mediana de 160 mm e varia de 70 a 320: um recorte de 160 mm
# e maior que a imagem em 60% das series e e pulado em todas elas, o que deixa a escala
# fisica delas nao normalizada. 130 mm fica abaixo do campo de visao de 99.6% das series
# e ainda contem a articulacao.
CROP_MM = 130.0

# Resolucao do cache. Tudo a jusante pode reamostrar para baixo a partir disto, entao e
# definida pela configuracao mais exigente, e nao pela padrao.
CACHE_IMG = 336
GROUP = 3                  # slices por input do encoder, empilhadas como os tres canais
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45      # fracao da memoria livre que o cache de pixels pode usar
CACHE_BUDGET_MAX_GB = 24.0 # teto rigido independente do que a maquina reporta
CACHE_BUDGET_GB = 12.0     # so o fallback, para uma maquina sem /proc/meminfo
TEST_SHARE = 0.30          # piso do corpus de teste relativo ao de treino, ja que o
                           # split de teste visivel e um stub e o avaliado nao e
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32         # ordenar slices e limitado por latencia no mount, nao por CPU
# Teto para a passada de ordenacao. Tem que ser um teto porque a passada e centenas de
# milhares de leituras pequenas sobre um mount de rede, entao sua duracao e uma
# propriedade do mount, nao do trabalho, e varia entre execucoes que fazem a mesma
# leitura. Nao pode ser um teto apertado: desistir deixa essas series em ordem de
# arquivo, o que nao tem correlacao com anatomia, e essa degradacao e silenciosa. Entao
# o teto fica bem acima do que a passada normalmente precisa: seu proposito e impedir
# que a passada consuma a execucao inteira num mount lento, nao aparar o caso comum, e
# um teto apertado o bastante para atuar num dia normal trocaria uma degradacao
# silenciosa por uma economia que a execucao nao precisa.
ORDER_BUDGET_S = 5400

# Resolucao e o eixo sob teste. Uma feature de largura d mm sobrevive a reamostragem so
# se o passo de pixel for no maximo d/2, e o passo aqui e definido pelo recorte acima,
# nao pelo campo de visao adquirido: CROP_MM / P. A 224 px isso e 0.58 mm, acima dos
# 0.5 mm que uma ruptura de 1 mm precisa; a 336 px e 0.39 mm e passa no teste. As duas
# configuracoes leem o mesmo cache, entao a comparacao isola o resize.
RUNS = [
    {"name": "r224", "img": 224},
    {"name": "r336", "img": 336},
]

EPOCHS = 10
BATCH_STUDIES = 8          # um estudo e um saco de ate N_SLOT imagens de slot
AUG_ROT_DEG = 8.0          # jitter rigido; ver augment() sobre por que nenhum flip e usado
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.10
LAT_MIN_OFFSET_MM = 20.0   # dentro disso o lado nao e legivel a partir da geometria; ver
                           # side_from_geometry()
SLICE_BAND = (0.20, 0.80)  # fracao da pilha ordenada sobre a qual read_slot amostra

# --- O que uma slice E, em oposicao a quantas delas existem --------------- #
#
# Um membro e uma funcao dos pixels sobre os quais foi ajustado, e img/crop_mm/slices/
# band nao determinam esses pixels sozinhos. Quatro decisoes a mais determinam, nenhuma
# delas visivel em nenhum shape:
#
#   order          qual slice vem a seguir ao longo da pilha
#   lat            quais joelhos sao espelhados, e com base em que evidencia
#   slot_fallback  se um slot T1 pode ser preenchido a partir de uma serie que nao e T1
#   decode_fill    o que substitui uma slice que nao decodificaria
#
# `native` e a leitura derivada nas secoes abaixo. `legacy` e a leitura sob a qual um
# membro importado foi ajustado. Um membro lido sob a errada carrega com todo shape
# batendo, roda, e escreve uma submissao plausivel calculada a partir da imagem errada -
# entao a escolha viaja junto com o membro e faz parte da chave que decide quais membros
# podem compartilhar uma decodificacao. As regras legadas sao reproduzidas em vez de
# corrigidas: corrigi-las entregaria a esse membro pixels que os pesos dele nunca viram.
RULES_NATIVE = {"order": "normal", "lat": "centre",
                "slot_fallback": False, "decode_fill": "nearest"}
RULES_LEGACY = {"order": "dominant_axis", "lat": "corner_x",
                "slot_fallback": True, "decode_fill": "zero"}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0   # a zona morta com a qual a regra legada de lateralidade foi ajustada

LR_HEAD = 1e-3
LR_BACKBONE = 8e-6         # o encoder e adaptado, nao retreinado do zero
UNFREEZE_LAST = 6          # blocos de transformer treinaveis, a partir do fim da saida
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600

# Seis slots: tres planos cruzados com os eixos de aquisicao. As series sensiveis a
# fluido com supressao de gordura existem para quase todo estudo; as series T1 e as
# sensiveis a fluido sem supressao sao mais escassas, e e para isso que serve a mascara
# de presenca.
SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]

# A alternativa: plano x o unico eixo que as flags entregues carregam, ignorando a
# ponderacao recuperada. Mantida como
# uma chave para que a escolha da definicao de slot possa variar enquanto tudo o mais
# fica fixo. Sob esse esquema um slot `Struct` mistura series T1 com series PD/T2 sem
# supressao de gordura, que carregam contraste de tecido muito diferente.
SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]

SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

# Quantas partes de largura 384 compoem a feature por slot. O encoder emite um vetor
# por token; uma feature de slot e um resumo fixo dessa grade, e o resumo com o qual um
# membro importado foi ajustado carrega uma terceira parte.
POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}

# Para quais slots a atencao de um membro importado se inclina, por diagnostico.
# Os indices sao em SLOTS. Esta e uma tabela fixa, nao um parametro aprendido, entao faz
# parte da definicao daquele membro e precisa ser reproduzida exatamente para que os
# pesos dele signifiquem algo.
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


In [ ]:
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def find_root():
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    # ultimo recurso: varredura de dois niveis, porque a montagem esta um nivel mais
    # funda que o usual
    base = Path("/kaggle/input")
    if base.is_dir():
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file():
                    return cand
    raise FileNotFoundError(
        f"competition mount not found (cwd {Path.cwd()}); expected a directory holding "
        f"test.csv and test_series/")


def find_dinov2(variant="small"):
    """Localiza um diretorio de checkpoint DINOv2 montado pelo nome da variante."""
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None


LABEL_COLS = TARGETS + [t + "__conf" for t in TARGETS]


class LabelSourceError(RuntimeError):
    """Levantada quando os rotulos nao vieram de onde esta execucao pretendia.

    Toda outra falha neste arquivo e melhor sobrevivida do que reportada: uma execucao
    que morre depois que o cache foi construido ja gastou a metade cara e nao pontua
    nada, entao a guarda em volta de `main` a engole e deixa o arquivo de benchmark para
    tras. Esta e a excecao. Treinar com os rotulos mais fracos nao parece uma falha -
    completa, escreve uma submissao plausivel, e difere apenas numa linha de log -
    entao tem que parar a execucao em vez de ser absorvida por uma guarda desenhada
    para crashes.
    """


def find_label_table():
    """Localiza uma tabela montada de rotulos de laudo pre-lidos, se alguma estiver anexada.

    O lexico transforma um laudo em rotulos casando morfologia, e seu modo de falha e o
    silencio: numa frase que ele nao cobre, emite nenhuma opiniao em vez de uma errada. O
    silencio e mensuravel sem nenhum gabarito - para cada par (laudo, achado), algo deu
    match? - e essa medida diz que as falhas se concentram em idiomas especificos em vez
    de se espalharem uniformemente, em achados que um laudo de joelho quase sempre
    comenta.

    Enumerar morfologia para nove idiomas e o instrumento errado para isso. Ler a frase
    e o certo, e um modelo de linguagem le. Contra os estudos anotados a diferenca e
    grande e unilateral, entao quando uma tabela dessas esta montada ela e preferida;
    quando nao esta, o lexico roda e o pipeline fica inalterado. Os dois caminhos
    produzem as mesmas colunas, entao nada a jusante sabe qual delas forneceu.
    """
    base = Path("/kaggle/input")
    cands = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            cands += [Path(root) / f for f in files if f.startswith("report_labels")
                      and f.endswith(".csv")]
    cands += [p for p in (Path("data/derived/report_labels_v2.csv"),) if p.is_file()]
    for c in cands:
        try:
            head = pd.read_csv(c, nrows=1)
        except Exception:
            continue
        if "StudyInstanceUID" in head.columns and all(t in head.columns for t in TARGETS):
            return c
    return None


def label_mount_attached():
    """True quando um diretorio de input foi anexado com a intencao de carregar uma
    tabela de rotulos.

    O fallback abaixo e deliberado e precisa ficar em silencio para uma execucao sem
    nenhuma tabela anexada, porque esse e o caso comum para quem le este notebook. Ele
    nao pode ficar em silencio para o outro caso: uma tabela foi anexada e nao pode ser
    usada. Os dois casos sao indistinguiveis so pelos rotulos - ambos terminam no lexico
    - entao eles sao separados aqui por existir ou nao a montagem em si.
    """
    base = Path("/kaggle/input")
    if not base.is_dir():
        return False
    return any("label" in p.name.lower() for p in base.iterdir() if p.is_dir())


def read_labels(train_df):
    """Rotulos para todo estudo de treino, de uma tabela montada ou do lexico.

    Estudos que a tabela montada nao cobre caem para o lexico em vez de serem
    descartados, entao uma tabela parcial degrada a cobertura em vez de perder linhas.
    """
    n = len(train_df)
    lab = pd.DataFrame([extract(r) for r in train_df["Report"].fillna("")])
    lab["StudyInstanceUID"] = train_df["StudyInstanceUID"].values
    lab = lab.set_index("StudyInstanceUID")

    src = find_label_table()
    if src is None:
        if label_mount_attached():
            raise LabelSourceError(
                "LABEL SOURCE: a label dataset is mounted but no usable table was found "
                "in it. Falling back to the lexicon here would train on the weaker "
                "labels and say so only in a log line, so the run stops instead.")
        log(f"LABEL SOURCE: lexicon, {n} studies (no table mounted)")
        return lab

    tab = pd.read_csv(src).set_index("StudyInstanceUID")
    missing = [c for c in LABEL_COLS if c not in tab.columns]
    if missing:
        raise LabelSourceError(
            f"LABEL SOURCE: {src} is missing {len(missing)} expected columns "
            f"(first: {missing[0]!r}). Refusing to fall back silently.")
    hit = lab.index.intersection(tab.index)
    if not len(hit):
        raise LabelSourceError(
            f"LABEL SOURCE: {src} shares no StudyInstanceUID with train.csv.")
    log(f"LABEL SOURCE: {src.name} covers {len(hit)} of {n} studies, "
        f"lexicon for the remaining {n - len(hit)}")
    lab.loc[hit, LABEL_COLS] = tab.loc[hit, LABEL_COLS].values
    return lab


ROOT = find_root()
log(f"input root: {ROOT}")


IMG = CACHE_IMG            # kept as the name the pixel reader and cache use


def available_gb():
    """Memoria que esta maquina de fato vai emprestar, lida em vez de suposta.

    Um teto fixo no codigo e um palpite sobre uma maquina em que o autor nao esta
    sentado, e um palpite baixo demais custa cobertura em silencio enquanto um alto
    demais encerra a execucao. A maquina vai dizer, entao ela e perguntada.
    """
    try:
        with open("/proc/meminfo") as fh:
            info = {k.strip(): v for k, v in
                    (l.split(":", 1) for l in fh if ":" in l)}
        return int(info["MemAvailable"].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION      # fall back to the old constant


def plan_cache(n_study, n_test=0):
    """Escolhe quantas slices por slot a memoria que a maquina tem vai permitir.

    O cache e n_study x n_slot x slices x IMG^2 bytes. Cobertura e o eixo barato -
    linear - e resolucao o caro, entao quando o orcamento aperta e a contagem de slices
    que cede, nao a grade de pixels. Decidir uma unica vez, a partir do tamanho do
    corpus de treino, mantem os caches de treino e teste no mesmo layout de grupo.

    So uma fracao do que esta livre e tomada. O resto nao e folga: o encoder, suas
    ativacoes, os batches fixados e os frames em transito saem todos do mesmo pool, e o
    cache e a unica alocacao grande o bastante para que estoura-la mate a execucao de
    vez.
    """
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    # Os dois caches sao mantidos ao mesmo tempo, e a metade de teste e o que a execucao
    # visivel nao consegue mostrar: aqui e um punhado de estudos, e na avaliacao e o
    # conjunto oculto inteiro. Dimensionar so pelo corpus de treino, portanto, passa em
    # toda execucao que pode ser observada e estoura justamente a que conta.
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f"memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; "
        f"sizing for {n_study} train + {n_total - n_study} test studies "
        f"-> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot"
        + (f" (wanted {N_GROUP_MAX})" if groups < N_GROUP_MAX else ""))
    return groups


N_GROUP = plan_cache(len(pd.read_csv(ROOT / "train.csv")),
                     len(pd.read_csv(ROOT / "test.csv")))
CACHE_SLICES = GROUP * N_GROUP
log(f"cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot")


In [ ]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Posicao e orientacao sao lidas do mesmo cabecalho que probe() ja
            # abre, entao nao custam nada, e sao o que recupera o lado quando a tag
            # Laterality esta ausente - o que acontece em metade dos estudos aqui.
            "ImagePositionPatient", "ImageOrientationPatient"]


def _hdr_vec(s, n):
    """Parse a DICOM multi-value string as stored by probe(): floats joined by `|`."""
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def side_from_geometry(h):
    """Estudo -> 'L' / 'R' / None, a partir de onde a imagem se situa no paciente.

    `Laterality` (0020,0060) e Tipo 2C e pode legitimamente estar ausente; neste corpus
    ela falta em exatamente metade dos estudos, e os fabricantes de onde falta sao
    fabricantes inteiros, nao series espalhadas. Um estudo sem a tag nao e um joelho
    esquerdo, mas a normalizacao a montante o trata como se fosse, entao metade do
    corpus nunca foi normalizada e os cinco alvos definidos por lado - os dois meniscos,
    os dois compartimentos tibiofemorais e o ligamento colateral medial - viram esse
    eixo invertido numa grande minoria dele.

    O sistema de coordenadas do paciente conserta isso sem a tag: +x e a esquerda do
    paciente, entao o centro de um joelho direito fica em x negativo. O centro e usado
    em vez do proprio `ImagePositionPatient` porque esse e o canto da imagem, deslocado
    por metade de um campo de visao - o suficiente para mudar o sinal num joelho perto
    da linha media.

    A mediana sobre as series de um estudo e o que se limiariza, nao uma unica serie:
    probe() le uma slice arbitraria por serie, que numa pilha sagital pode estar em
    qualquer lugar da articulacao. Estudos cujo centro cai perto da linha media ficam
    sem resolucao em vez de serem chutados - medida contra a metade com tag, a regra
    acerta 97% das vezes no geral e nao melhor que o acaso dentro de 20 mm.
    """
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def side_from_corner_x(h):
    """A lateralidade sob a qual um membro importado foi ajustado.

    Ela limiariza a mediana do x bruto de `ImagePositionPatient` sobre as series de um
    estudo. Esse e o x do *canto* da imagem, nao do seu centro, entao difere da regra
    acima em ate meio campo de visao - o suficiente para inverter o sinal num joelho
    escaneado perto da linha media. A zona morta e 5 mm em vez de 20 mm, entao ela
    tambem se compromete em estudos que a regra acima deixa sem resolucao.

    Nenhuma das duas diferencas muda um shape. Cada uma decide se um estudo e
    espelhado, e um estudo espelhado de um jeito no treino e de outro na inferencia
    apresenta os cinco alvos definidos por lado com o eixo invertido.
    """
    out = {}
    for st, g in h.groupby("StudyInstanceUID"):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        # Coordenadas de paciente em DICOM sao LPS: +x e a esquerda do paciente.
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else ("R" if x < 0 else "L")
    return out


def lat_of(h, tag=""):
    """Estudo -> 'L' / 'R' / None: a tag onde ela existe, geometria onde nao existe.

    A tag esta presente em exatamente metade dos estudos aqui e as vezes e uma string
    vazia em vez de ausente, o que nao e a mesma coisa que NaN. Tratar a outra metade
    como do lado esquerdo e o que `normalise_laterality` fazia por omissao, entao o
    fallback de geometria nao e um refinamento - e a diferenca entre normalizar metade
    do corpus e normalizar ele inteiro.
    """
    geo = side_from_corner_x(h) if RULES["lat"] == "corner_x" else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        if RULES["lat"] == "corner_x" and "ImageLaterality" in g.columns:
            # A regra legada le a segunda tag tambem, entao um estudo marcado so ali e
            # resolvido a partir da tag, nao da geometria.
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, "
        f"{n_none} unresolved; tag and geometry disagree on {n_disagree} "
        f"({n_disagree / max(n_tag, 1):.1%} of the tagged)")
    return d



def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    """Todo diretorio de serie de um split, com uma leitura de cabecalho por serie.

    Um split ausente retorna um frame vazio *com as colunas que annotate espera*.
    Retornar um DataFrame nu parece a mesma coisa e nao e: a proxima chamada indexa
    `SeriesDescription` e levanta KeyError, entao o ramo que existe para sobreviver a
    um split ausente e o que o transforma num crash.
    """
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recupera a supressao de gordura e a ponderacao de sequencia de pulso a partir do
    cabecalho."""
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # A GE escreve SAT_GEMS para saturacao espacial, entao ScanOptions precisa ser
    # casado como tokens exatos; um teste de substring em "SAT" dispara em series que
    # nao sao fat-sat.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


In [ ]:
def pick_slots(series_df, plane_map):
    """Uma serie por slot por estudo.

    Empates sao resolvidos a favor da pilha com mais slices: uma pilha mais espessa
    amostra a articulacao mais densamente, e o amostrador de tres slices abaixo se
    beneficia dessa margem.
    """
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None significa "nao condicionar na ponderacao" - o esquema publico,
            # onde a unica flag fornecida substitui os dois eixos de uma vez.
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            # Um slot sem nenhuma serie que case seu predicado fica vazio, e nenhum
            # substituto e admitido de um predicado vizinho. Relaxar a ponderacao para
            # preencher um slot T1 puxaria do mesmo pool de onde `SAG_FLUID_NOFS`
            # seleciona, ja que esse pool e o que sobra quando a ponderacao e descartada:
            # no corpus de treino isso colocaria uma serie em dois slots para 2383 de
            # 4407 estudos e deixaria 56% do slot T1 contendo PD ou T2. A mascara de
            # presenca entao afirmaria uma sequencia que nunca foi adquirida, e o softmax
            # por diagnostico da secao 6 dividiria sua atencao entre dois slots
            # identicos, dando a uma aquisicao cerca do dobro do peso que ela carrega
            # num estudo que tem as duas. A mascara existe para dizer que um slot esta
            # ausente, que e o que um slot ausente e.
            if len(cand) == 0 and RULES["slot_fallback"] and fluid is False:
                # O relaxamento que o paragrafo acima rejeita, reproduzido porque um
                # membro importado foi ajustado com seus slots T1 preenchidos desse
                # jeito: mais da metade dos estudos de treino desse membro tinha um slot
                # T1 contendo uma serie que nao e T1. Deixar esses slots vazios o
                # apresentaria a uma mascara de presenca que ele nunca viu.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out


## 3b. Em que ordem as slices estao

Uma serie e um diretorio de arquivos, e o jeito obvio de percorre-lo e ordenar os nomes de
arquivo. Isso esta errado aqui, e errado de um jeito que nao produz erro nenhum. O nome do
arquivo e o SOP Instance UID, atribuido para ser unico e nao para ser ordenado, entao ordenar
por ele produz uma sequencia sem correlacao com a anatomia -- medido numa serie deste corpus,
a correlacao de rank entre a ordem por nome de arquivo e a posicao fisica e $\rho \approx
0.01$. Tres coisas a jusante dependem dessa ordem: tres slices adjacentes como tres canais
viram tres cortes transversais nao relacionados; o meio da pilha vira um subconjunto
aleatorio; e inverter a ordem para normalizar lateralidade nao faz absolutamente nada.

A ordem verdadeira e recuperavel exatamente a partir da geometria que toda slice carrega.
`ImageOrientationPatient` da os eixos no plano $\hat{r}_x, \hat{r}_y$ e
`ImagePositionPatient` a posicao $p$ do primeiro voxel, entao

$$\hat{n} \;=\; \hat{r}_x \times \hat{r}_y, \qquad k \;=\; p \cdot \hat{n},$$

e $k$ aumenta monotonicamente ao longo da pilha. Como $k$ tem sinal e e expresso em
coordenadas do paciente, a pilha tem uma direcao fixa ao longo do eixo esquerda-direita do
corpo -- que e o que a normalizacao de lateralidade da secao 5 inverte. `InstanceNumber` e o
fallback onde as tags de geometria estao ausentes; normalmente acompanha $k$ a menos de
sinal, mas aquisicoes intercaladas e multi-eco numeram as slices numa ordem que nao e a ordem
que elas ocupam no espaco, e nao tem sinal em coordenadas do paciente.


## 4. Amostragem: quantos milimetros um pixel pode valer

Uma slice DICOM de $N \times N$ pixels com espacamento $s$ mm/pixel cobre $Ns$ milimetros de
anatomia. As duas variam ao longo deste corpus, entao um resize de pixel fixo entrega ao
encoder imagens cuja escala fisica difere por um fator de varias vezes. Mas a normalizacao de
escala e so metade da questao; a outra metade e um limite rigido.

**Uma feature mais estreita que dois pixels nao sobrevive ao resize.** Para representar uma
estrutura de largura $d$ milimetros o passo de pixel precisa satisfazer $s_{\text{eff}} \le
d/2$, a condicao de Nyquist aplicada a grade de reamostragem. Uma ruptura meniscal tem de um a
tres milimetros, entao com $d = 1$ mm o passo precisa ser no maximo $0.5$ mm -- e se nao for,
nenhuma capacidade a jusante recupera o sinal, porque ele foi destruido antes da primeira
convolucao. Isso e uma propriedade do resize, nao da rede.

Recortar para uma extensao fisica constante $L$ e reamostrar para $P$ pixels fixa o passo:

$$n \;=\; \Big\lfloor \frac{L}{s} \Big\rceil \ \text{pixels}, \qquad
s_{\text{eff}} \;=\; \frac{L}{P}\ \ \text{mm/pixel}, \qquad
\text{token} \;=\; 14\,s_{\text{eff}}\ \ \text{mm}.$$

O recorte precisa ser menor que o menor campo de visao ou ele nao faz nada em silencio: se
$L/s$ excede a largura da imagem o recorte nao pode ser tomado e aquela serie passa sem ser
normalizada, sem erro nenhum. $L = 130$ mm fica abaixo do campo de visao adquirido de quase
toda serie aqui, mas ainda contem a articulacao. O alvo de resize entao decorre da largura da
ruptura, e nao de convencao -- com $L = 130$ mm um input de $224$ da $0.580$ mm/pixel, acima
do limite para uma feature de 1 mm, enquanto $336$ da $0.387$ mm/pixel e coloca um token de
patch de $14$ pixels em $5.4$ mm.

**Intensidade precisa do mesmo tratamento**, ja que RM nao tem escala absoluta. Cada serie e
normalizada pelo seu proprio percentil 1 e 99 -- sobre a pilha amostrada, e nao por slice, para
que as slices mantenham seu contraste relativo, e por percentis em vez de extremos, para que
um vaso brilhante nao comprima todo o resto.


In [ ]:
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

# Series nas quais ao menos uma slice amostrada nao decodificaria. Uma lista em vez de
# um contador porque adicionar e atomico sob as threads leitoras, e reportado em vez de
# engolido: sem reportar, uma falha de decode e indistinguivel de um joelho preto.
DECODE_FAILED = []


def cache_tag(rules=None):
    """O nome sob o qual um cache decodificado e armazenado.

    Ele precisa nomear tudo que decide os pixels, nao so as dimensoes deles. Duas
    configuracoes que concordam em resolucao, contagem de slices, recorte e faixa mas
    discordam em como uma slice e escolhida produzem arrays diferentes de shape
    identico - entao uma tag construida so a partir das dimensoes deixa a segunda se
    anexar ao arquivo da primeira e treinar contra pixels que nunca pediu, sem nada em
    lugar nenhum reportando a incompatibilidade.

    Uma leitura nativa mantem o nome simples, entao caches decodificados antes das
    regras existirem continuam validos; qualquer outra coisa ganha um sufixo.
    """
    r = dict(RULES if rules is None else rules)
    t = (f"{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_"
         f"{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}")
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += "_" + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t


def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))


def _order_dominant_axis(rec):
    """A ordem de slice sob a qual um membro importado foi ajustado.

    Ela ordena pela coordenada de paciente bruta ao longo de qualquer que seja o eixo
    que mais varia na pilha, em vez de pela projecao sobre a normal da slice. As duas
    diferem por um sinal, nao por uma formula: medido neste corpus toda serie sagital
    tem uma normal de slice com n_x em [-1.00, -0.98], entao p.n e o negativo do x bruto
    sobre o qual isto ordena e as duas pilhas saem exatamente invertidas. Como o
    amostrador de faixa trunca em vez de arredondar, seus nove indices nao sao
    simetricos em torno do meio, entao nove slices tiradas de uma pilha de vinte e seis
    sob uma ordem compartilham duas com a outra.

    Geometria ausente cai para `InstanceNumber` e depois para uma ordenacao natural do
    nome de arquivo, ambas no mesmo limiar de 80% que o pipeline importado usava.
    """
    files, d = rec["files"], rec["dir"]
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=["ImagePositionPatient", "InstanceNumber"])
            raw = getattr(ds, "ImagePositionPatient", None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, "InstanceNumber", None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))

    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare,
                                 r[2] if r[2] is not None else float("inf"), r[3]))
    elif sum(r[2] is not None for r in rows) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float("inf"), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return [r[0] for r in rows], True


def order_slices(rec):
    """Retorna os arquivos da serie ordenados ao longo do eixo through-plane.

    Um nome de arquivo DICOM aqui e um SOP Instance UID, atribuido arbitrariamente.
    Ordenar por ele, portanto, produz uma ordem sem correlacao com a anatomia - medido
    numa serie, o Spearman entre o rank do nome de arquivo e a posicao fisica e 0.009,
    ou seja, nenhuma. Qualquer coisa que assuma que a ordem do arquivo significa algo
    esta entao operando sobre ruido: os tres canais de um input "2.5D" sao tres vistas
    nao relacionadas em vez de slices vizinhas, "o meio da pilha" e um subconjunto
    aleatorio, e inverter a ordem das slices para normalizar lateralidade nao inverte
    nada significativo.

    A ordem fisica e recuperavel exatamente. Cada slice carrega sua posicao em
    coordenadas de paciente e os eixos no plano; projetar a posicao sobre a normal da
    slice da uma coordenada through-plane com sinal, monotonica ao longo da pilha:

        n = r_x  x  r_y ,      k = p . n

    `InstanceNumber` e o fallback. Normalmente acompanha a projecao a menos de sinal,
    mas aquisicoes intercaladas e multi-eco nao precisam numerar as slices na ordem que
    ocupam no espaco - mas a projecao tem sinal em coordenadas de paciente, que e o que
    a normalizacao de lateralidade precisa.
    """
    if RULES["order"] == "dominant_axis":
        return _order_dominant_axis(rec)
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None for k, _ in keyed):
        # Uma serie sem geometria utilizavel mantem sua ordem arbitraria; isso e pior
        # que ordenar mas melhor que descartar a serie, e e registrado como uma contagem.
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` slices fisicamente espalhadas de uma serie, em `out_size` pixels.

    Retorna uint8 [n_slice, out, out] normalizado por-serie pelo seu percentil 1-99.
    Percentis em vez de min/max porque a intensidade em RM nao tem escala absoluta e um
    unico vaso brilhante, caso contrario, comprimiria toda a faixa dinamica.

    Ler e a metade cara deste pipeline, entao quem chama le uma vez na maior
    configuracao de que precisa e deriva as menores a partir do buffer retornado em vez
    de reler.
    """
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Espalha as amostras sobre uma faixa central da pilha: as slices mais externas de
    # uma serie de joelho sao principalmente tecido mole fora da articulacao. A faixa e
    # uma constante em vez de um literal porque quanto da pilha vale a pena ler depende
    # de quantas slices estao sendo tomadas - com tres, so o meio cabe, enquanto com
    # dezesseis as pontas valem a pena, e um cisto de Baker fica na ponta posteromedial
    # de uma pilha sagital.
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None                      # no shape is known here; see below
        planes.append(a)

    # Uma slice que nao decodificaria nao tem shape proprio, e inventar um e como um
    # unico arquivo ilegivel apaga uma serie inteira: um substituto alocado no alvo de
    # resize enquanto as slices decodificadas ainda estao nativas faz a checagem de
    # shape abaixo tomar o substituto como autoridade e zerar as slices boas junto com
    # ele, deixando um slot preto que a mascara de presenca ainda reporta como
    # adquirido.
    #
    # Uma falha e, em vez disso, preenchida a partir da slice mais proxima que de fato
    # decodificou - a mesma convencao que o amostrador ja usa quando a faixa contem
    # menos slices distintas do que foram pedidas - e uma serie onde nada decodifica e
    # reportada como ausente, que a mascara consegue expressar, em vez de preta, que ela
    # nao consegue.
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES["decode_fill"] == "zero":
        # Aquilo com que um membro importado foi ajustado: uma falha vira um plano zero
        # no alvo de resize, que a checagem de shape abaixo entao propaga para o slot
        # inteiro. E o comportamento que o paragrafo acima descreve e rejeita, mantido
        # aqui so porque os pesos daquele membro foram aprendidos contra slots
        # apagados dessa forma.
        if not got:
            DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p
                  for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    # Slices de uma mesma serie ainda podem diferir em tamanho de matriz - multi-eco e
    # alguns reformats diferem - e essas genuinamente nao sao empilhaveis.
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # extensao fisica constante, depois resize: PixelSpacing varia 3.4x ao longo do corpus
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, nao float32. Esses buffers se enfileiram entre as threads leitoras e o
    # encoder, e neste tamanho uma slot-serie em float32 tem varios megabytes. A
    # intensidade ja esta normalizada em [0, 1] aqui, entao oito bits nao custam nada
    # que um resize bilinear ja nao tenha custado, e a fila fica com um quarto do
    # tamanho.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


## 5. Normalizando esquerda e direita

Quatro dos doze alvos -- os dois meniscos e os compartimentos tibiofemorais medial e lateral --
sao pares medial/lateral, e um quinto, o ligamento colateral medial, e nomeado pelo lado em que
fica. Medial e lateral sao definidos em relacao a linha media do corpo, entao de que lado da
*imagem* eles caem depende de qual joelho foi escaneado. A menos que isso seja normalizado,
esses cinco rotulos aprendem a partir de um eixo que o modelo nao consegue observar.

A correcao difere por plano. Coronal e axialmente a direcao medial-lateral esta no plano da
imagem, entao inverter o ultimo eixo mapeia um joelho no outro. Sagitalmente e o eixo da
*slice*: cada slice fica inalterada pelo espelhamento, e o que difere e a ordem em que a
pilha percorre a articulacao, entao a ordem das slices e invertida em vez disso.

`Laterality` e um atributo Tipo 2C: pode legitimamente estar ausente, e aqui esta ausente em
metade dos estudos -- por fabricantes inteiros, nao series espalhadas. Deixar esses estudos
como estao declara silenciosamente que sao do lado esquerdo, entao todo joelho direito entre
eles entra no modelo espelhado. O sistema de coordenadas do paciente fornece a tag ausente:
posicao e orientacao sao registradas por imagem e $+x$ aponta para a esquerda do paciente,
entao o sinal do $x$ do centro da imagem diz qual joelho e este.

$$c \;=\; \mathbf{p} \;+\; \mathbf{r}\,\Delta_c \frac{N_c}{2} \;+\; \mathbf{d}\,\Delta_r \frac{N_r}{2},
\qquad \text{lado} = \begin{cases} \text{direito} & c_x < 0\\ \text{esquerdo} & c_x > 0\end{cases}$$

com $\mathbf{p}$ a posicao da imagem, $\mathbf{r}$ e $\mathbf{d}$ os cossenos diretores de
linha e coluna, e $\Delta$ o espacamento de pixel. O centro e usado em vez do proprio
$\mathbf{p}$, que e um canto a meio campo de visao de distancia -- o suficiente para mudar o
sinal num joelho escaneado perto da linha media. A mediana sobre as series de um estudo e o
que se limiariza, e um estudo centrado a uma curta distancia da linha media fica sem resolucao
em vez de ser chutado, porque dentro dessa faixa o sinal nao e melhor que o acaso.


In [ ]:
def normalise_laterality(img, plane, lat):
    """Mapeia todo joelho numa convencao de joelho esquerdo.

    Vistas coronais e axiais espelham sob um flip horizontal. Pilhas sagitais nao sao
    imagens espelhadas uma da outra - a ordem das slices corre de medial para lateral em
    direcoes opostas - entao a ordem dos canais e invertida em vez disso.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


## 5b. Lendo uma vez, treinando muitas vezes

O custo deste pipeline e dominado pela leitura, nao pela aritmetica. Um estudo tem varias
series e cada serie dezenas de slices, entao um estudo fica na ordem de cento e cinquenta
arquivos. Isso e pagavel uma vez; nao e pagavel a cada epoca, e o fine-tuning precisa dos
mesmos pixels a cada epoca. Entao as imagens de slot sao decodificadas uma unica vez para a
memoria e mantidas como `uint8`:

$$\text{bytes} \;=\; N_{\text{estudo}} \times N_{\text{slot}} \times S \times P^{2}$$

com $S$ slices mantidas por slot em $P$ pixels. O expoente em $P$ e o que torna isso uma
restricao real, e nao um detalhe -- o cache cresce com o *quadrado* da resolucao e so
linearmente com as slices, entao cobertura e o eixo barato e resolucao o caro.

As slices em cache formam um input de encoder de tres canais por slot, e o layout se
generaliza para varios desses grupos por slot: o treino sorteia um por passo, o que funciona
como uma augmentacao ao longo da pilha, e a inferencia faz a media entre eles. O que fixa o
numero de grupos e um orcamento, e nao uma capacidade -- o cache pode usar uma fracao da
memoria reportada como livre, deliberadamente abaixo do total, ja que ele e a unica alocacao
grande o bastante para que estoura-la encerre a execucao em vez de so deixa-la mais lenta, e o
encoder, suas ativacoes e os buffers em transito vem do mesmo pool. O tamanho comparado
contra esse orcamento e a soma dos *dois* caches, treino e teste, porque os dois ficam
residentes ao mesmo tempo.

Um arquivo de submissao valido e escrito antes de qualquer coisa disso comecar e sobrescrito
so quando predicoes de verdade existem, entao a execucao sempre deixa para tras um arquivo
avaliavel.


In [ ]:
# Onde a ordem geometrica de slice pode ser lembrada entre execucoes. Nao definida na
# plataforma, porque cada execucao ganha uma maquina nova e nao ha nada para lembrar;
# definida fora dela, onde o mesmo corpus e colocado em cache de novo a cada resolucao
# e contagem de slices e a ordem nao e funcao de nenhuma das duas. E opt-in para que o
# comportamento da execucao avaliada seja decidido pelo codigo, e nao por um arquivo
# que por acaso esteja por ai.
ORDER_CACHE = os.environ.get("RSNA_ORDER_CACHE") or None


def build_cache(slot_map, plane_map, lat_map, tag):
    """Decodifica cada (estudo, slot) uma vez para um array uint8 em memoria.

    O fine-tuning revisita os mesmos pixels a cada epoca. Le-los do mount toda vez
    tornaria a contagem de epocas uma funcao de I/O em vez de aprendizado, entao eles
    sao decodificados uma vez e mantidos como bytes: a intensidade ja foi normalizada
    em [0, 1], e oito bits nao custam nada que um resize bilinear ja nao tenha custado.

    CACHE_SLICES posicoes sao mantidas por slot, que o loop de treino le como N_GROUP
    grupos de GROUP canais consecutivos.
    """
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    n_job = len(jobs)

    # Ordenar primeiro, e como sua propria passada. Le um cabecalho por slice de toda
    # serie escolhida - muito mais aberturas de arquivo que o decode que vem depois - e
    # num mount de rede isso e latencia, nao trabalho, entao ganha seu proprio pool mais
    # largo.
    t_ord = time.time()
    n_slice_total = sum(len(j[3]["files"]) for j in jobs)
    log(f"{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)")
    ok = done = 0
    CHUNK_O = 1024

    # Uma ordem lembrada, quando uma e oferecida. A projecao depende so da geometria
    # DICOM, entao e a mesma em toda resolucao e toda contagem de slices, e custa uma
    # leitura de cabecalho por slice - o maior custo unico nesta passada. Uma entrada e
    # validada pelo numero de arquivos presentes, entao uma arvore que mudou por baixo
    # dela e recalculada em vez de confiada: ordem e dado derivado, e uma entrada velha
    # seria invisivel do jeito que mais importa.
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec["SeriesInstanceUID"])
            if e and len(e["files"]) == len(rec["files"]):
                rec["ordered"] = e["files"]
                ok += int(e["good"])
                hit += 1
        jobs = [j for j in jobs if "ordered" not in j[3]]
        log(f"{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read")

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(
                    block, pool.map(lambda j: order_slices(j[3]), block)):
                rec["ordered"] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec["SeriesInstanceUID"]] = {"files": files, "good": bool(good)}
            # O teto e o que vier primeiro: o orcamento proprio da passada, ou a fatia
            # do que resta da execucao que ela pode tomar. O segundo e o que torna
            # seguro definir o primeiro generosamente - um mount lento o bastante para
            # importar nao consegue gastar o tempo de treino, porque o orcamento encolhe
            # conforme a execucao encolhe.
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f"{tag}: ordering budget spent at {done}/{len(jobs)}; "
                    f"the rest keep file order")
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix(".tmp")
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f"{tag}: ordered {ok}/{n_job} by geometry "
        f"({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")
    n_failed_before = len(DECODE_FAILED)

    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(
                    block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > TIME_BUDGET:
                log(f"  {tag}: time budget reached during decode")
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f"{tag}: {int(mask.sum())}/{len(jobs)} slots filled"
        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))
    gc.collect()
    return studies, cache, mask


## 6. Agregando slots em doze decisoes

Um estudo chega como ate seis embeddings de slot $x_s \in \mathbb{R}^{d}$ com uma mascara de
presenca $m_s \in \{0,1\}$. Fazer pooling deles de forma identica descartaria o motivo pelo
qual o protocolo tem tres planos: cada achado e lido em sequencias particulares, e uma media
sobre os slots dilui aquele que carrega a evidencia com cinco que nao carregam.

Projete cada slot, some uma identidade de slot aprendida, de a cada diagnostico $o$ sua
propria query $q_o \in \mathbb{R}^{H}$, e deixe-o dar atencao sobre os slots com os ausentes
mascarados fora do softmax:

$$h_s \;=\; \phi(x_s) + e_s, \qquad
\alpha_{o,s} \;=\; \frac{\exp\!\big(\langle h_s, q_o\rangle / \sqrt{H}\big)\, m_s}
{\sum_{s'} \exp\!\big(\langle h_{s'}, q_o\rangle / \sqrt{H}\big)\, m_{s'}},$$

$$c_o \;=\; \sum_s \alpha_{o,s}\, h_s, \qquad
\ell_o \;=\; \langle c_o, w_o \rangle + b_o .$$

O softmax mascarado renormaliza sobre o que o estudo de fato contem, entao uma serie axial
ausente desloca a atencao de um diagnostico para as sequencias que estao presentes em vez de
alimenta-lo com um vetor zero.

**A cabeca de saida e deliberadamente pequena assim.** O rotulo esta atrelado ao *estudo*,
entao nada na supervisao diz qual parte de um estudo carrega o achado, e uma agregacao mais
rica nao teria sinal nenhum para aprender isso. Onde a supervisao e grosseira, a cabeca
tambem e.


In [ ]:
class SlotHead(nn.Module):
    """Atencao por diagnostico sobre os embeddings de slot de um estudo.

    Cada achado e lido em sequencias particulares - cruzados sagitalmente, ligamentos
    colaterais e o corpo do menisco coronalmente, cartilagem patelar axialmente - entao
    fazer pooling dos slots de forma identica diluiria aquele que carrega a evidencia com
    o resto.

    A agregacao e deliberadamente simples assim. Com um rotulo em nivel de estudo nao ha
    sinal dizendo ao modelo qual parte de um estudo importa, entao parametros de atencao
    extras abaixo do nivel do slot nao teriam nada para aprender e gastariam sua
    capacidade ajustando ruido.
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        # Um membro importado carrega uma inclinacao fixa por par (diagnostico, slot) nos
        # logits de atencao, definida a partir da tabela de anatomia abaixo em vez de
        # aprendida. E um buffer, entao viaja no state dict e precisa existir para que
        # aquele membro carregue; exp(0.55) da a um slot preferido cerca de 1.73x o peso
        # de um nao preferido, o que enviesa o softmax sem nunca excluir um slot.
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and n_out == len(TARGETS):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


In [ ]:
class Model(nn.Module):
    """Encoder mais cabeca de saida, treinados de ponta a ponta.

    Um estudo chega como um saco de imagens de slot. O saco e achatado para o encoder e
    dobrado de volta antes da cabeca de saida, entao o encoder nunca ve a estrutura do
    estudo e a cabeca nunca ve pixels.
    """

    def __init__(self, backbone, dim, pool="cls_mean", prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            # O cache e mantido na maior resolucao que qualquer configuracao precisa; o
            # resto reamostra para baixo a partir dele, entao toda configuracao ve os
            # mesmos pixels atraves de uma grade de amostragem diferente, em vez de um
            # recorte diferente.
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear",
                              align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            # A cauda superior de cada canal sobre a grade de patches, tomada por canal em
            # vez de selecionando patches inteiros: um achado ocupa uma pequena parte do
            # campo, entao uma media simples sobre 256 patches o dilui em duas ordens de
            # grandeza, e isto mantem o oitavo superior das respostas de cada canal.
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)


### Por que o encoder e treinado em vez de congelado

Um encoder auto-supervisionado congelado e limitado por algo que nenhum trabalho a jusante
consegue alcancar. Resolucao, tamanho do encoder, cobertura de slices e agregacao de slots
mudam quanto o modelo olha e com que atencao, mas nenhum muda o vocabulario com que ele olha,
entao todos esses eixos esbarram no mesmo teto -- e deveria se esperar que esse teto atue
aqui, ja que o encoder aprendeu suas features a partir de imagens naturais, onde nada se
parece com o sinal que uma ruptura de menisco faz numa sequencia ponderada em densidade de
protons.

Entao o encoder e adaptado, com duas restricoes. **So os ultimos blocos se movem** -- os
blocos iniciais de um transformer de visao sao filtros genericos de borda e textura, os
blocos finais e onde a semantica mora, e pode nao haver supervisao suficiente aqui para
melhorar os iniciais enquanto certamente ha o bastante para danifica-los. **O encoder aprende
bem mais devagar que a cabeca de saida** -- a cabeca e aleatoria na inicializacao e tem tudo a
aprender, o encoder parte de uma boa solucao e so precisa ser deslocado dela, entao uma unica
taxa de aprendizado ou deixaria a cabeca sem treino ou destruiria o encoder nos primeiros
poucos centenas de passos.

Os alvos continuam sendo os rotulos derivados de laudo da secao 2, ponderados pela confianca
por achado que vem junto com eles, entao um laudo que nunca menciona um achado puxa fraco
naquela saida em vez de afirmar um negativo ali. Os estudos que carregam anotacoes por
condicao sao ponderados acima de toda linha derivada, ja que sao os unicos rotulos lidos das
imagens.


In [ ]:
def build_model(unfreeze_last, source=None, variant="small", pool="cls_mean",
                prior=False):
    """Carrega o encoder e abre os ultimos `unfreeze_last` blocos para treino.

    Os blocos iniciais de um transformer auto-supervisionado sao filtros genericos de
    borda e textura; os blocos finais carregam semantica. Abrir so os finais e a escolha
    cautelosa - pode nao haver supervisao suficiente aqui para melhorar os iniciais e
    certamente ha o bastante para danifica-los - mas onde exatamente essa linha deve
    ficar e uma pergunta que o corpus tem que responder, nao a intuicao.

    `source` nomeia de onde vem os pesos. Deixado sem definir, e o diretorio de modelo
    anexado, que e a unica coisa disponivel aqui. E um parametro para que uma execucao
    fora da plataforma construa o mesmo objeto a partir do mesmo codigo, em vez de uma
    segunda definicao que precisaria ser mantida sincronizada a mao.
    """
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError("DINOv2 weights not attached")
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    log(f"backbone: {n_layer} blocks, last {unfreeze_last} trainable "
        f"({trainable / 1e6:.1f}M params), feature dim {dim * POOL_PARTS[pool]}")
    return Model(bb, dim, pool=pool, prior=prior)


## 6b. Lendo pesos em vez de aprende-los

Nada exige que a execucao avaliada seja a execucao que aprendeu os pesos. Um notebook pode
anexar um dataset, pesos sao um dataset, e a parte que genuinamente nao pode ser feita com
antecedencia e a parte que depende de estudos que ninguem viu -- le-los, e predizer. Aprender
aqui, em vez disso, limita o modelo ao que um unico acelerador cabe dentro do limite de
tempo, e gasta esse tempo de novo a cada submissao para um resultado que nao muda. Entao,
quando um pacote de membros treinados esta anexado, esta secao o le; quando nenhum esta, as
secoes abaixo treinam um.

**Membros, nao um modelo.** Cada membro carrega o pre-processamento sob o qual foi ajustado.
Dois membros ajustados em resolucoes diferentes nao podem compartilhar um decode; dois
ajustados de forma igual podem. Entao os membros sao agrupados pelos pixels de que precisam,
cada grupo e decodificado uma vez, e um membro adicionado depois entra na lista sem
alteracoes.

O que os agrupa e mais que a resolucao e o recorte. Quatro decisoes a mais determinam o que
uma slice *e* -- qual vem a seguir ao longo da pilha, quais joelhos sao espelhados, se um
slot pode ser preenchido a partir de uma sequencia que nao casa com seu predicado, e o que
substitui uma slice que nao vai decodificar -- e nenhuma delas muda um unico shape. Um membro
ajustado sob uma leitura e decodificado sob outra, portanto, carrega normalmente, roda, e
retorna uma submissao calculada a partir da imagem errada, entao a leitura viaja junto com o
membro.

**Um membro precisa provar que e o modelo que era.** Carregar um state dictionary tem
sucesso sempre que os shapes batem, e os shapes batem em toda diferenca que importa -- uma
normalizacao mudada, um resize, uma faixa de slice. Nenhuma delas levanta erro. Entao cada
membro carrega a resposta que deu a uma pergunta com semente fixa, recalculada antes do uso;
uma incompatibilidade para a execucao em vez de ser diluida numa submissao.

**Como um membro e lido.** O treino sorteia um grupo de slices consecutivas por passo. Na
inferencia, olhar mais nao custa decodificacao extra -- o cache ja mantem $S$ slices, entao
um grupo do meio e $1$ passagem forward, os grupos disjuntos sao $S/G$, e toda sequencia
consecutiva de $G$ slices e $S-G+1$. Onde a media e tomada e gratis mas nao e neutro: fazer a
media dos logits e depois esmagar e uma media geometrica das chances (odds), fazer a media
das probabilidades e uma media aritmetica do risco, e elas ordenam os estudos de forma
diferente. Os membros sao combinados por rank, ja que a secao 1 estabeleceu que ordem e tudo
que a metrica le.


In [ ]:
FINGERPRINT_TOL = 2e-3


def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    """A saida do modelo sobre um saco sintetico fixo, como uma identidade portavel.

    Pesos que sao carregados mas lidos atraves do pre-processamento errado produzem
    predicoes, nao erros. A submissao fica bem formada, o log nao diz nada, e a
    diferenca e um numero que nenhuma saida da execucao revela. Um escalonamento que
    nunca acontece, ou que acontece duas vezes, basta sozinho e nao muda shape nenhum
    em lugar nenhum.

    Entao um conjunto de pesos carrega a resposta que deu a uma pergunta sem nenhum
    dado nela. O input e gerado a partir de uma semente em vez de lido, entao e o mesmo
    em qualquer maquina, e e empurrado por todo o caminho forward - o escalonamento de
    bytes, a normalizacao ImageNet, o resize, o encoder, a atencao de slot. Qualquer uma
    dessas coisas diferindo move a saida por ordem um. Numerica diferindo entre duas
    GPUs move por cerca de 1e-5, por isso a tolerancia fica entre os dois em vez de em
    zero.

    Isso checa que o modelo computa o que computava quando foi ajustado. Nao pode
    checar que os pixels que chegam a ele sao os pixels certos; `read_slot` e a passada
    de cabecalho respondem por seus proprios testes.
    """
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size),
                         generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0                       # exercita o ramo mascarado do softmax
    was_training = model.training
    model.eval()
    with torch.no_grad():
        # float32 do inicio ao fim: autocast faria o valor depender de qual dispositivo
        # por acaso o rodou, e o ponto do numero e justamente nao depender.
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out


def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=""):
    """Compara contra uma fingerprint armazenada; levanta erro quando o modelo nao e o
    mesmo mapeamento."""
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f"{tag}fingerprint shape {got.shape} != stored {exp.shape}: "
                           f"the architecture is not the one these weights were fitted to")
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(
            f"{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load "
            f"but do not compute what they computed when fitted - preprocessing, "
            f"resolution or architecture has moved between the two runs.")
    log(f"{tag}fingerprint matches within {d:.2g}")
    return d


class WeightsError(RuntimeError):
    """Levantada quando pesos anexados nao podem ser confiados como sendo os que foram
    ajustados.

    Deliberadamente fatal pelo mesmo motivo que LabelSourceError: uma execucao que
    prediz a partir de um modelo incompativel completa, escreve uma submissao
    plausivel, e difere de uma correta so num numero que nenhuma saida da execucao
    revela.
    """


def find_weights(name="manifest.json"):
    """Localiza um pacote de pesos montado, ou retorna None se nenhum estiver anexado.

    Mesmo formato de `find_label_table`: o notebook precisa continuar funcionando para
    um leitor que nao anexa nada, entao ausencia e um caminho, nao um erro. O que nao
    pode ficar em silencio e um pacote que esta anexado e inutilizavel, e e isso que
    `load_weights` recusa.
    """
    import json
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if name not in files:
            continue
        # O manifesto decide, nao os nomes de arquivo ao lado. Testar por uma convencao
        # de nomenclatura faz a busca concordar com o que quer que o empacotador tenha
        # chamado seus arquivos por ultimo, o que e uma segunda definicao do que um
        # pacote e.
        try:
            man = json.loads((Path(root) / name).read_text())
        except (OSError, ValueError):
            continue
        if isinstance(man.get("members"), list) and man["members"]:
            missing = [m["file"] for m in man["members"]
                       if not (Path(root) / m["file"]).is_file()]
            if missing:
                raise WeightsError(
                    f"{root} holds a manifest listing {len(man['members'])} members but "
                    f"{len(missing)} of their files are absent (first {missing[0]!r})")
            return Path(root)
    return None


# Como um membro e lido na inferencia. Janelas sobrepostas sobre as slices que o cache
# ja mantem custam passagens forward e nenhuma decodificacao extra, que e a direcao
# barata para gastar; e fazer a media das probabilidades em vez dos logits e uma media
# aritmetica do risco em vez de uma media geometrica das chances (odds), o que ordena
# os estudos de forma diferente. Ambos foram escolhidos medindo-os nos folds que cada
# membro deixou de fora, e nao por argumento.
TTA_OVERLAP = True
TTA_POOL = "prob"

# Ajuste incorporado do fork da comunidade (renta0426/rsna-knee-baseline-v1-
# fracture-tta-pool-probe, publicScore 0.893 vs 0.891 da v15 original):
# em vez de MEDIA das janelas de TTA pra todo mundo, usa MAXIMO nessas duas
# classes -- fratura e ruptura de menisco lateral costumam aparecer numa
# janela especifica da pilha de slices; media dilui o sinal com janelas que
# nao veem o achado, maximo captura a janela que efetivamente viu.
# Labels nao listados aqui mantem a media original, sem mudanca.
TTA_TARGET_POOL = {
    "Fracture": "max",
    "Lateral Meniscus": "max",
}


def window_starts(n_slice, group, overlap=None):
    """Onde cada janela de TTA comeca."""
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]


@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None,
                   starts=None):
    """As predicoes de um membro, agregadas sobre suas janelas de TTA."""
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)

    if not starts:
        raise ValueError("predict_member was given no windows to average over")

    target_idx = {t: j for j, t in enumerate(TARGETS)}
    unknown = set(TTA_TARGET_POOL) - set(target_idx)
    if unknown:
        raise ValueError(f"unknown target(s) in TTA_TARGET_POOL: {unknown}")

    model.eval()
    out = []

    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)

        acc = None
        win_logits = []
        win_probs = []

        for st in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(cache[sel, :, st:st + group])
            ).to(dev)

            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                z = model(rows, m, img_size).float()

            p = torch.sigmoid(z)
            base = z if pool == "logit" else p
            acc = base if acc is None else acc + base

            if TTA_TARGET_POOL:
                win_logits.append(z)
                win_probs.append(p)

        # Caminho original (media) -- continua valendo pra quem nao esta
        # em TTA_TARGET_POOL.
        v = acc / len(starts)
        if pool == "logit":
            v = torch.sigmoid(v)

        # Sobrescreve so os targets explicitamente listados.
        if TTA_TARGET_POOL:
            probs = torch.stack(win_probs, dim=0)    # [window, batch, target]
            logits = torch.stack(win_logits, dim=0)

            for target, mode in TTA_TARGET_POOL.items():
                j = target_idx[target]

                if mode == "mean":
                    v[:, j] = probs[:, :, j].mean(dim=0)
                elif mode == "logit_mean":
                    v[:, j] = torch.sigmoid(logits[:, :, j].mean(dim=0))
                elif mode == "max":
                    v[:, j] = probs[:, :, j].max(dim=0).values
                elif mode in ("top2", "top3"):
                    k = min(int(mode[3:]), probs.shape[0])
                    v[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(dim=0)
                else:
                    raise ValueError(f"unknown TTA pooling mode for {target}: {mode}")

        out.append(v.cpu().numpy())

    return (
        np.concatenate(out)
        if out
        else np.zeros((0, len(TARGETS)), np.float32)
    )


def infer_from_package(path, dev):
    """Prediz o split de teste a partir de um pacote anexado de membros treinados.

    A alternativa abaixo - aprender os pesos dentro da execucao avaliada - gasta toda
    a cota no corpus de treino toda vez que o notebook e submetido, e limita o modelo
    ao que nove horas num unico acelerador conseguem caber. Nenhuma das duas e
    necessaria: um notebook pode anexar um dataset, e pesos sao um dataset. O que a
    execucao avaliada entao faz e a parte que nao pode ser feita com antecedencia,
    porque os estudos nao sao conhecidos com antecedencia.

    Membros sao agrupados pelos pixels de que precisam. Dois membros ajustados em
    resolucoes diferentes sao funcoes diferentes do mesmo estudo e nao podem
    compartilhar um decode; dois ajustados de forma igual podem, e esse e o motivo
    inteiro pelo qual o agrupamento existe, em vez de um decode por membro.
    """
    import json
    man = json.loads((Path(path) / "manifest.json").read_text())
    members = man["members"]
    log(f"weights package: {len(members)} member(s) from {path}")

    test_df = pd.read_csv(ROOT / "test.csv")
    test_series = pd.read_csv(ROOT / "test_series.csv")
    plane_map = dict(zip(test_series["SeriesInstanceUID"],
                         test_series["Anatomical_Plane"]))
    hte = annotate(walk("test_series"))
    log(f"test header pass: {len(hte)} series")

    groups = {}
    for m in members:
        groups.setdefault(m["pixel_group"], []).append(m)

    per_member = []
    fixed_s = per_win_s = None
    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, "
            f"crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map,
                                      lat_of(hte, "test "), f"test g{gi}")
        idx = np.arange(len(st_te))

        # O que o tempo restante permite. Um membro custa uma parte fixa - carregar o
        # checkpoint, construir o encoder, checar a fingerprint - mais uma parte que
        # escala com o numero de janelas de TTA sobre as quais e lido. So a segunda vale
        # a pena negociar, e so o primeiro membro consegue medir qualquer uma das duas,
        # porque o tamanho do conjunto de teste nao e algo que o notebook recebe de
        # antemao.
        #
        # As duas partes sao cronometradas separadamente. Dividir o tempo de parede
        # inteiro de um membro pela sua contagem de janelas precifica a parte fixa como
        # se ela escalasse, entao toda reducao infla a estimativa que a causou e a
        # estimativa se afasta da verdade.
        #
        # Janelas sao entregues antes dos membros. Uma janela descartada custa uma vista
        # de um estudo que outras janelas tambem veem; um membro descartado custa um
        # voto independente, que e do que um ensemble e feito. A pergunta feita antes de
        # cada membro e se MAIS UM cabe - nao se todos os outros cabem, o que abandonaria
        # um ensemble que ainda poderia ter rodado a maior parte de si mesmo.
        starts = window_starts(Cte.shape[2], GROUP)
        order = sorted(gm, key=lambda m: -(m.get("holdout") or 0))
        left_after = sum(len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi)
        for k, m in enumerate(order):
            left = TIME_BUDGET - (time.time() - T0)
            remaining = (len(order) - k) + left_after
            if fixed_s is not None and per_win_s is not None:
                # Deixa um decimo do que resta sem gastar: a estimativa vem de um
                # membro numa maquina, e uma submissao escrita tarde nao e escrita.
                afford = max(left * 0.9, 0.0)
                need = fixed_s + len(starts) * per_win_s
                if need * remaining > afford:
                    room = afford / max(remaining, 1)
                    n_win = int((room - fixed_s) / per_win_s) if per_win_s > 0 else 0
                    n_win = max(1, min(len(starts), n_win))
                    if fixed_s + per_win_s > afford:
                        log(f"  {left / 60:.0f} min left: stopping after {k} of "
                            f"{len(order)} in this decode group; not one more member "
                            f"fits, at any window count")
                        break
                    if n_win < len(starts):
                        mid = (len(starts) - n_win) // 2
                        log(f"  {left / 60:.0f} min left, {remaining} member(s) to go: "
                            f"{n_win} window(s) each instead of {len(starts)}")
                        starts = starts[mid:mid + n_win]
            t0 = time.time()
            ck = torch.load(Path(path) / m["file"], map_location="cpu",
                            weights_only=False)
            model = build_model(int(m["config"]["unfreeze_last"]),
                                variant=m["config"]["variant"],
                                pool=m["config"].get("pool", "cls_mean"),
                                prior=bool(m["config"].get("prior", False))).to(dev)
            model.load_state_dict(ck["model"])
            check_fingerprint(model, dev, IMG, ck["fingerprint"], tag=f"{m['id']}: ")
            t_ready = time.time()
            p = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts)
            per_member.append({"id": m["id"], "ids": st_te, "pred": p,
                               "holdout": m.get("holdout")})
            fixed_s = t_ready - t0
            per_win_s = (time.time() - t_ready) / max(len(starts), 1)
            log(f"  {m['id']} fold {m['fold']}: predicted {len(idx)} studies over "
                f"{len(starts)} window(s) in {time.time() - t0:.0f}s")
            del model, ck
            gc.collect()
            if dev.type == "cuda":
                torch.cuda.empty_cache()
        del Cte, Mte
        gc.collect()

    # Rank em vez de probabilidade, porque a metrica le ordem e dois membros calibrados
    # de forma diferente, do contrario, teriam peso desigual. Todo membro cobre todo
    # estudo de teste, entao a media e sobre o mesmo conjunto sempre e nao precisa de
    # ponderacao para ser comparavel; ponderar pelo holdout de um fold importaria para
    # o conjunto de teste uma diferenca medida em algumas centenas de estudos de
    # treino.
    all_ids = sorted({s for m in per_member for s in m["ids"]})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    for m in per_member:
        r = pd.DataFrame(m["pred"]).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m["ids"]]] += r
    acc /= max(len(per_member), 1)

    sub = write_submission(acc, all_ids, test_df, "submission.csv")
    log(f"submission.csv = rank mean of {len(per_member)} member(s); {sub.shape}; "
        f"nulls {int(sub[TARGETS].isna().sum().sum())}")
    return sub


def adopt_config_globals(cfg):
    """Aponta o caminho de pixels para aquilo sobre o qual um grupo de membros foi
    ajustado."""
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg["img"])
    GROUP = int(cfg["group"])
    CACHE_SLICES = int(cfg["slices"])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg["crop_mm"])
    SLICE_BAND = tuple(float(x) for x in cfg["band"])
    # As quatro decisoes que mudam o que uma slice e. Um membro ajustado sob uma leitura
    # e decodificado sob outra recebe pixels que seus pesos nunca viram, com todo shape
    # ainda batendo, entao um nome nao reconhecido e recusado em vez de receber um
    # padrao.
    rules = cfg.get("rules") or RULES_NATIVE
    unknown = {k: v for k, v in rules.items()
               if k not in RULES_NATIVE
               or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f"the members record pixel rules this pipeline cannot "
                           f"reproduce: {unknown}")
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg["slots"]):
        raise WeightsError(
            f"the members were fitted on slots {cfg['slots']} and this pipeline defines "
            f"{[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")


In [ ]:
def take_group(cache_rows, g):
    """Recorta GROUP canais consecutivos das slices em cache."""
    return cache_rows[:, :, g * GROUP:(g + 1) * GROUP]


def augment(imgs):
    """Um pequeno jitter rigido e uma escala de intensidade, aplicados a um saco inteiro de
    uma vez.

    Nenhum dos dois flips esta disponivel aqui, e por motivos diferentes. Um flip
    horizontal reintroduziria o eixo de ruido que a normalizacao de lateralidade
    removeu - desfaria, uma vez por batch, o que a passada de cabecalho foi executada
    para estabelecer.

    Um flip vertical nao e um eixo de ruido de jeito nenhum. Um joelho e adquirido numa
    orientacao canonica, e nenhum estudo neste corpus se parece com seu proprio espelho
    vertical. Uma augmentacao deve cobrir direcoes ao longo das quais o rotulo nao muda;
    esta move o input para fora da distribuicao sobre a qual o encoder sera questionado,
    o que e uma coisa diferente. Onde um achado se situa no quadro tambem e informacao,
    e nao ruido - um cisto de Baker e identificado por estar na fossa popliteal, nao so
    pela aparencia.

    O que sobra e jitter do qual nenhum rotulo depende: alguns graus de rotacao, alguns
    por cento de escala e translacao. Isso ainda impede memorizar o enquadramento
    exato, que e para o que uma augmentacao serve, deixando a anatomia onde estava.
    """
    # Um saco chega como [study, slot, GROUP, IMG, IMG]: cinco eixos, nao quatro. O warp
    # e uma operacao 2-D, entao os dois eixos iniciais sao dobrados juntos e restaurados
    # depois - cada imagem de slot e uma aquisicao independente e ganha seu proprio
    # jitter.
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = x.shape[0], x.device

    rot = (torch.rand(n, device=dev) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    # So zoom in. O padding `border` repete a linha da borda para fora, e a borda deste
    # recorte e onde fica a fossa popliteal; dar zoom out fabricaria tecido exatamente
    # onde um cisto de Baker e procurado.
    sc = 1.0 + torch.rand(n, device=dev) * AUG_SCALE
    tx = (torch.rand(n, device=dev) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev) - 0.5) * 2 * AUG_SHIFT
    cos, sin = torch.cos(rot) / sc, torch.sin(rot) / sc
    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = cos, -sin, tx
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sin, cos, ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)

    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)


@torch.no_grad()
def predict(model, cache, mask, idx, dev, img_size=None):
    """Faz a media dos logits sobre os grupos de cada slot.

    O treino ve um grupo por vez, o que age como augmentacao ao longo da pilha; a
    inferencia faz a media sobre todos eles, entao a predicao nao depende de qual
    grupo um unico sorteio por acaso escolheu. Onde o cache mantem um grupo por slot os
    dois coincidem.
    """
    model.eval()
    out = []
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        acc = None
        for g in range(N_GROUP):
            # Coletado um grupo por vez em vez do estudo inteiro depois recortado. Os
            # dois sao os mesmos pixels, mas tirar o estudo inteiro do cache aloca toda
            # slice que ele contem - a maioria das quais esta passada nao vai olhar ate
            # uma iteracao posterior, quando ja tiverem sido buscadas de novo. Medido
            # sobre um cache de doze slices, a diferenca entre os dois e a diferenca
            # entre o passo ser limitado por memoria ou ser limitado pelo encoder.
            rows = torch.from_numpy(np.ascontiguousarray(
                cache[sel, :, g * GROUP:(g + 1) * GROUP])).to(dev)
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                z = model(rows, m, img_size).float()
            acc = z if acc is None else acc + z
        out.append(torch.sigmoid(acc / N_GROUP).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)


def macro_auc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(np.nanmean([roc_auc_score(y[:, j], p[:, j])
                             if len(set(y[:, j])) > 1 else np.nan
                             for j in range(y.shape[1])]))


## 7. Validando sem se enganar

Dois vazamentos sao especificos deste setup, e os dois inflam um numero de validacao sem
melhorar nada.

**Laudos compartilhados.** Alguns laudos sao identicos byte a byte entre estudos -- um modelo
lido para um joelho sem alteracoes -- entao todo estudo nesse grupo recebe o mesmo
vetor-alvo derivado, e dividir o grupo entre os dois lados avalia o modelo num alvo cuja
fonte ele ja viu no treino. Os estudos sao, portanto, atribuidos por um hash do texto do
laudo, o que mantem todo grupo de duplicatas inteiro. Um quinto fica de fora como holdout, e
a divisao e fixa, nao rotativa.

**Duas referencias, dois significados.** O **holdout** cobre um quinto do corpus, mede a
concordancia com os alvos derivados, e tem estudos suficientes por rotulo para separar uma
diferenca real de ruido -- ele seleciona tanto a epoca dentro de uma execucao quanto a receita
entre execucoes. A **checagem de anotacao** mede a concordancia com a leitura de um
radiologista das imagens, que e o que a competicao pontua, mas so os estudos anotados que
caem no holdout podem ser usados e ha muito poucos; pelo argumento de erro-padrao da secao 2,
ela e reportada e nunca autorizada a arbitrar.

Os estudos anotados permanecem no treino, com peso elevado, porque sao os unicos rotulos
lidos das imagens em vez do texto. E exatamente por isso que a checagem de anotacao precisa
ficar restrita ao holdout: avaliar um modelo em exemplos de treino cujas respostas ele viu,
ponderados mais pesadamente que qualquer outra coisa, mede memorizacao e a reporta como
habilidade.


In [ ]:
def write_submission(pred, studies, test_df, path):
    """Escreve um arquivo de submissao a partir de uma matriz de predicoes.

    As predicoes sao convertidas em ranks por coluna primeiro: a metrica so le ordem,
    entao ranks nao descartam nada, e tornam arquivos de configuracoes diferentes
    diretamente comparaveis e seguros de promediar.
    """
    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)
    sub.insert(0, "StudyInstanceUID", studies)
    sub = test_df[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub


def write_benchmark_submission():
    """Escreve imediatamente o arquivo de benchmark com 0.5.

    Uma submissao que nunca e escrita nao pontua absolutamente nada, o que e
    estritamente pior que pontuar mal. O try/except em volta de main() cobre excecoes,
    mas um kill por memoria e um SIGKILL e nunca chega ate ele. Entao um arquivo valido
    existe desde o primeiro segundo e so e sobrescrito quando predicoes de verdade
    estao prontas.
    """
    t = pd.read_csv(ROOT / "test.csv")
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv("submission.csv", index=False)


def main():
    write_benchmark_submission()

    # Pesos, se algum foi anexado; caso contrario a execucao aprende os proprios abaixo.
    # Os dois caminhos sao mantidos porque o segundo e o que torna este notebook legivel
    # por si so - um fork sem nada anexado ainda treina e ainda pontua - e porque o
    # primeiro nao pode ser checado por quem nao tem o pacote.
    pkg = find_weights()
    if pkg is not None:
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        infer_from_package(pkg, dev)
        log("done")
        return

    # Resolve de onde vem os rotulos antes de qualquer coisa cara rodar. A checagem
    # custa a leitura de um cabecalho de CSV; descobrir o mesmo problema depois que o
    # cache foi construido custaria a passada de decode inteira, e nunca descobrir
    # custaria a execucao inteira.
    read_labels(pd.read_csv(ROOT / "train.csv", usecols=["StudyInstanceUID", "Report"]))

    test_df = pd.read_csv(ROOT / "test.csv")
    test_series = pd.read_csv(ROOT / "test_series.csv")
    train_df = pd.read_csv(ROOT / "train.csv")
    train_series = pd.read_csv(ROOT / "train_series.csv")
    log(f"train {train_df.shape} test {test_df.shape}")

    both = pd.concat([train_series, test_series])
    plane_map = dict(zip(both["SeriesInstanceUID"], both["Anatomical_Plane"]))

    log("header pass: test")
    hte = annotate(walk("test_series"))
    log(f"  {len(hte)} test series")
    log("header pass: train")
    htr = annotate(walk("train_series"))
    log(f"  {len(htr)} train series")

    slots_te, slots_tr = pick_slots(hte, plane_map), pick_slots(htr, plane_map)
    cov = pd.Series([len(v) for v in slots_tr.values()]).describe()
    log(f"train slots per study: mean {cov['mean']:.2f} min {cov['min']:.0f} "
        f"max {cov['max']:.0f}")

    st_tr, Ctr, Mtr = build_cache(slots_tr, plane_map, lat_of(htr, "train "), "train")
    st_te, Cte, Mte = build_cache(slots_te, plane_map, lat_of(hte, "test "), "test")

    # ---- targets ---------------------------------------------------------- #
    t_lab = time.time()
    lab = read_labels(train_df)
    log(f"derived labels for {len(lab)} studies in {time.time() - t_lab:.1f}s")

    gold = train_df.set_index("StudyInstanceUID")[TARGETS]
    gold = gold[gold.notna().all(axis=1)]

    Y = np.zeros((len(st_tr), len(TARGETS)), np.float32)
    W = np.zeros_like(Y)
    for i, st in enumerate(st_tr):
        if st in gold.index:
            Y[i], W[i] = gold.loc[st].values, 3.0
        elif st in lab.index:
            r = lab.loc[st]
            Y[i] = r[TARGETS].values
            W[i] = 0.25 + 0.75 * r[[t + "__conf" for t in TARGETS]].values
    keep = np.where(W.sum(1) > 0)[0]
    log(f"supervised {len(keep)} of {len(st_tr)} studies (annotated {len(gold)})")

    # Agrupado pelo texto do laudo: alguns laudos sao identicos byte a byte entre
    # estudos e produzem um unico vetor-alvo para todos eles, entao dividir esse grupo
    # avalia o modelo num alvo cuja fonte ele ja treinou.
    import hashlib
    rep = train_df.set_index("StudyInstanceUID")["Report"].fillna("")
    grp = np.array([int(hashlib.md5(rep.get(s, s).encode()).hexdigest()[:8], 16) % 5
                    for s in st_tr])
    va = np.array([i for i in keep if grp[i] == 0])
    tr = np.array([i for i in keep if grp[i] != 0])
    if len(va) == 0 or len(tr) < BATCH_STUDIES:
        cut = max(1, len(keep) // 5)
        va, tr = keep[:cut], keep[cut:]
    log(f"train {len(tr)} / holdout {len(va)} studies")

    # Os estudos anotados ficam no treino - sao os rotulos de mais alta qualidade no
    # corpus e sao poucos demais para descartar - entao a checagem de anotacao honesta
    # usa so os que cairam no holdout. Avaliar sobre o resto seria pontuar o modelo
    # contra exemplos com os quais ele treinou, com peso triplo, com a resposta
    # verdadeira.
    gpos = {s: i for i, s in enumerate(st_tr)}
    va_set = set(va.tolist())
    gi = np.array([gpos[s] for s in gold.index if s in gpos and gpos[s] in va_set])
    gold_y = gold.loc[[st_tr[i] for i in gi]].values.astype(int) if len(gi) else None
    yv = (Y[va] > 0.5).astype(int)
    log(f"annotation check: {len(gi)} of {len(gold)} annotated studies are in the holdout")

    # ---- fine-tune -------------------------------------------------------- #
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    results, test_preds = {}, {}

    for cfg in RUNS:
        pitch = CROP_MM / cfg["img"]
        log(f"=== {cfg['name']}: {cfg['img']} px, {pitch:.3f} mm/pixel, "
            f"{pitch * 14:.2f} mm per patch token ===")
        torch.manual_seed(SEED)
        model = build_model(UNFREEZE_LAST).to(dev)
        opt = torch.optim.AdamW([
            {"params": [p for p in model.backbone.parameters() if p.requires_grad],
             "lr": LR_BACKBONE},
            {"params": model.head.parameters(), "lr": LR_HEAD},
        ], weight_decay=WEIGHT_DECAY)
        steps = max(EPOCHS * (len(tr) // BATCH_STUDIES), 1)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=[LR_BACKBONE, LR_HEAD], total_steps=steps, pct_start=0.15)
        scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

        best, best_state, best_annot = -1.0, None, float("nan")
        for ep in range(EPOCHS):
            model.train()
            perm = np.random.permutation(tr)
            tot, nstep = 0.0, 0
            for b in range(0, len(perm) - BATCH_STUDIES + 1, BATCH_STUDIES):
                sel = perm[b:b + BATCH_STUDIES]
                rows = torch.from_numpy(Ctr[sel]).to(dev)
                g = int(torch.randint(N_GROUP, (1,)).item())
                imgs = augment(take_group(rows, g))
                m = torch.from_numpy(Mtr[sel]).to(dev)
                y = torch.from_numpy(Y[sel]).to(dev)
                w = torch.from_numpy(W[sel]).to(dev)
                with torch.autocast("cuda", enabled=dev.type == "cuda"):
                    loss = (F.binary_cross_entropy_with_logits(
                        model(imgs, m, cfg["img"]), y, reduction="none") * w).mean()
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
                tot += loss.item()
                nstep += 1

            pv = predict(model, Ctr, Mtr, va, dev, cfg["img"])
            d = macro_auc(yv, pv)
            g_auc = float("nan")
            if gold_y is not None and len(gi):
                g_auc = macro_auc(gold_y, predict(model, Ctr, Mtr, gi, dev, cfg["img"]))
            log(f"  epoch {ep + 1}/{EPOCHS}  loss {tot / max(nstep, 1):.4f}"
                f"  holdout {d:.4f}  annot(n={len(gi)}) {g_auc:.4f}")

            # A selecao le so o holdout. A checagem de anotacao e reportada porque mede
            # algo diferente - concordancia com uma leitura das imagens em vez dos
            # laudos - mas so um punhado de estudos anotados cai em qualquer holdout,
            # entao seu erro de amostragem ofusca as diferencas entre epocas e ela nao
            # pode arbitrar entre elas.
            if d > best:
                best, best_annot = d, g_auc
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            if time.time() - T0 > TIME_BUDGET:
                log("  time budget reached")
                break

        if best_state is not None:
            model.load_state_dict(best_state)
        results[cfg["name"]] = (best, best_annot)
        test_preds[cfg["name"]] = predict(model, Cte, Mte, np.arange(len(st_te)), dev,
                                          cfg["img"])
        log(f"  {cfg['name']}: best holdout {best:.4f} (annot {best_annot:.4f})")
        del model, opt, sched, scaler, best_state
        gc.collect()
        if dev.type == "cuda":
            torch.cuda.empty_cache()

    log("---- summary ----")
    for n, (d, g_auc) in results.items():
        log(f"  {n:12s} holdout {d:.4f}   annot {g_auc:.4f}")
    pick = max(results, key=lambda k: results[k][0])
    log(f"best on the holdout: {pick} ({results[pick][0]:.4f})")


    # ---- escreve todo candidato -------------------------------------------- #
    # Um arquivo por configuracao, mais a escolha do holdout como `submission.csv`. Uma
    # execucao custa um decode completo do corpus seja qual for a configuracao
    # vencedora, entao manter todo braco torna uma mudanca de configuracao posterior
    # gratuita em vez de outra execucao completa.
    for name, pred in test_preds.items():
        sub = write_submission(pred, st_te, test_df, f"submission_{name}.csv")
        log(f"  submission_{name}.csv {sub.shape}; "
            f"nulls {int(sub[TARGETS].isna().sum().sum())}")

    ens = np.mean([pd.DataFrame(p).rank(pct=True).values for p in test_preds.values()],
                  axis=0)
    write_submission(ens, st_te, test_df, "submission_rankmean.csv")
    log(f"  submission_rankmean.csv (rank mean of {len(test_preds)})")

    sub = write_submission(test_preds[pick], st_te, test_df, "submission.csv")
    log(f"submission.csv = {pick}; {sub.shape}; "
        f"nulls {int(sub[TARGETS].isna().sum().sum())}")
    print(sub.head().to_string())


In [ ]:
try:
    main()
except LabelSourceError:
    # Deliberadamente nao absorvida: ver LabelSourceError. Uma execucao que treinou com
    # os rotulos errados terminaria e escreveria uma submissao que pareceria valida o
    # bastante para ser submetida por engano.
    traceback.print_exc()
    raise
except Exception:
    traceback.print_exc()
    # Uma submissao que falha ao escrever nao pontua absolutamente nada, entao volta
    # para o arquivo de benchmark em vez de morrer.
    t = pd.read_csv(find_root() / "test.csv")
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv("submission.csv", index=False)
    print("wrote fallback submission.csv")
log("done")
